# ARC Prize 2026 — ARC-AGI-3 Submission

Built from `agent/my_agent.py` via `scripts/build_notebook.py`. Do not edit cells directly — edit the source file and re-run `make submit`.

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /tmp/my_agent.py
"""Agent ARC-AGI-3 « System-2 » (projet JEPA) : un modèle du monde JEPA VIDÉO (mémoire des derniers pas de la partie) imagine
plusieurs pas ; un planificateur fait descendre le GRADIENT d'une note imaginée (critique « fin de niveau proche », nouveauté,
effet, souvenir des réussites, danger) sur des séquences d'actions assouplies (Mode 2 de LeCun), puis joue la 1re action.
Mémoire des essais (pas deux fois le même geste dans le même état), clics accrochés aux objets, apprentissage pendant la partie
(renforcé au début). Pas de repli sur le v1. Arrêt des parties vers 8 h (limite 9 h)."""
from __future__ import annotations
from typing import Any
import base64, glob, os, sys, threading, time, zlib
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

FILES = {
    "explorer.py": "eNqlWd1uG8cVvtdTnNJAtetQNGk7bkGFBmSbkRW4liDJTgqCIIa7Q3LD5ex6f2QpqoHe5r5Ab3PXqH2Confhm/RJ+p2Z2T+STp1kYVjcmTNnznznf7bVau0dHQ9fX9LR+fODo+OTg0d01aP//vVv9NO/KJQ0Xd/hH3l5Esj8mn76DzlxEigviGVKfk5xEn0rM/pqeHbUpliklGI8jgI/dam/R9Sjs+H58+Hb05NzejG8oNNnXw0vL6hPXrSKo1SoDPRepJS8ZoaSVut/riRG8lDmCTmhoFgmnoyzIFI0X99Fq/VdlgTvckn+vphGid+GnOt/YPYsTxcHly4dYl/CIBiHgZeSknQVpFJlpFdhPJpC6JQcD4MJM8qVrEnktulbsRJBSrmiOLiWIYmcFiIVid8B84e0/v7yCMcYULASczC8OHp9wXAxj4wF7+v9w2CuZPrAi0I+oQbs5ZsX5ExFordd3ymZzAPZrhbiUB6fFfh5C6HmLPb6B+AsU0ivT4ZxPgijffCUEhmv75L1HdhLZXYkaGQlQM+jbUoDBXTsKi0wtJSIIGMdqyi/kiFEh2qxLNyX13EYJULjnUV5ogwpeCeR0sd/RMNvzl6dng/P6ejt8Dk9f3N+cnpxcrn+vm2AOD66ODt59QrzfZonIl5IgtJExuKa0/EvM3RYqFED5ujBtqVyCzXINBU367u0rc/vCwVo97GGQQToUis23BcaLUFHZ2fnJxeUAk5magElJxMwYX9fzmY4aSwSym5iWSJOD0qz8/e1hbD9MIjakonROnp+eXL6+hHNGJJplM9lwnuHvL+3iFKGsc2EeqXk/QEbyNgI9SL4kQIRrBQLhfIZY6CbZsUhKfCCNgUhFGvg13s73kKuAsX84jBnl8mTzCIhLcguXcmEkTHIslfl5QI4KlhgzwAASeVFBrtwH87MhKzYx3R5Dv19OTy/JMSE8yG9Pnk7PHrzDfQIB1DBlcSfNA+u4CXtamO2QcHiSPoOthiRVppWyz4OkvK5/Zoi+LRz+NL6jnc9H14ML80YzC4HraMKT2IjRyChPAvCIAU4Ik7WP6awsD8NJ6dvYWGHWsVZAiQCw12DUBGkkWJvSIw3MGRQmswRpzp7JwoyxFJhMGPnnCViJd9HyRJbDrUfyKQDqZ15EiDOpIBVThRo+PBwm3TCYoYyk5gVVyIIxTSUk9KDYeWOeZnwel9kwu3stRB0A8QaIIaYsgiD6d4siVY4MvzQQmTnfQkrwkI5E3mY+YGXFStVvopvCCFAxXt7z06OEYseU/nc48hl4seKfcK3VoC/TKtjGhxiEQEqRCQXqzneF54gjK1uWvbe3h4kMYFSgW3qzNs0nU8Amzfodh5+3gbQ1xM1ePxH17gMjmpj7aAR8R8f/EzM11LNEGw43oNe4zhy7HybMuAcAhXnpk3XLkdH623uuMPY8sYv2/Q19px30oWI5SECHtxuALA638kkSh2HCRDnp1EUuofEPjig0VivnUUJ3XQJ3pYwgM5Le5hi7ro293Vtjp9gprca3XQhW3fMeU5lgbKRu3igaRauoEIo4N0d8+qODxs8MHWZ5DgDtFYJWTzvFwH88F1TCn4YHJC/68RR7Lh6dUfEbOyOAc7dWqIPznPXfECQ0WfUY1LGmg6q38waU+VvTLnutggWkC59MQBf+oJeEkc8/Y49voCK+F1FmT2w3nusB+fl24BNJ9zNnZ/G0gKrd9VZ9czGaSFVKJUDTFx6Wpgw3YeA9+nrSmkf2fGets4GwzNjXMjr4kazPSQPG3usgrPOSgrldDG25JJBZY7jnAE0x5C4Lt2/Tw/dTpqvnJ4LJnOEemdDZJhoeSZmAUzcdnkKaIIHz0bLNnXHeC3femPXtayYRYrg4SzlzSAUq6kvKOpTNAIN/YLnHmqyLMjKOqrI4A4yYobohUowWP/oydRsm6vgXZuwZ6rNF3FUZo5bulrExoaFffr1j4l3qPZ0WjfJ2w9SKBERsyBaMvpONOpChGj0cIwf9OABPbRvPfNWoQ4jwRK2TgjI4vf1/x3h+85yCW3ywQqdRGYdkjZytp4Z9XUwHCNqeqFI0zKjGFPmSDqZBCrIJhMnleGMcZH+oMvFoEhSOejWXIoJOomaGzND7PGjVcemhQnGHV7rHho6s16HXP3DQmTZekhuukJIi/qVztffv3hzcjmERQou4L313z3Y0Ezouvo4OrCiN8VBHQUOtdzkGJvq06jXwTHwn7WrexuVFsdzNS1KMdiLeUEBFKRjqAjV/grak1x1cIZGceE291by/USnYOc1UtHm5ARrYz5ed2/P7H9QPsg2d6sosGW/LWpKjVR8jUo48W+qgXPt7YdPtMuy8L01Z7fYF1UU/YX4AOiqRFnmuhhsHZ+2PjS39aD09NO2rm072ixA2loX7njDtKL3E2/B5sXilHYUbg+qSTQ10DYYoN9YTsAlrWfZbplgKxow3UnTZId+56q5bxwKpc0Nmd7ZoIZ7ZZb6/0Oz1WNEXOLD0lgFKC/vvPWdj+pmrzSKPPa58FvkvrUKgDitWQWKDtPp6RbC9npcaZcdnMONpMzdWitXb+MGZf8Hxhu9ISuxbObK+sbGJ2HqG/rdgKbmZ9+GoJLKB3ehCfYaRUpN6SjWNXQIxvVxw5CjJFb75VszE29YT6nWit7dsqYtol5J9BHrMsOfDai322wxw8x0/kRC77r0+9oAyo1Hm1JUK7qbK7p2xRZeRoqnA3qyA4O6AzRke9BYTN3OH3a4Q0O27RWVLSIDFaGpZoELXet6UXzjbItdSlYpdFCWxqxQWoyahG3qjxtKaLBikXey6mlW/fYG7W5WNjc9pV7/VyV8+Ck3fWC2muq2i2E0nQT330J3LHwP07jRQBNxlei8EgYy5+YDdCq6kUmzQK9UwlIeaoBhtugAvaUzwiH7HkDqeyglFiNPFw56qPw5HleKsEWBbfY6K/9zh2vFVJeZ8zzKU1M3LtxOFk1v0CCh+Oss5LUfzNEXQ6Wl+jkJBByN0sIKbONZMwZPNxOIUezUyOKto1vxoeXqYktwLaMX6PBRVDcchZ/UcgLmnpSUTWNnLtAqXyt9VzVhgWq0hTuaAa+sXp9g1XWbbrjCaD2/BTNIt4WWVx06ToIosedF8qoxhwCqUBVqiRFm0T1ZDhKOpHYUAByLRVpy5/6+Sva2yy8a/G10bQATVmdmjc85dYBhlN29J00XZHYa49br08vJ2aujPw9ftNrI8MUVRWtHD6kXwbVqVLotqtIdgjYrj3W8DTbLXeDC1DbiHI/S8UiwP3J9sR3Danm0lnU7XihF4hQRtJaaGxws6t32J+Xh3U5t7oIcexHUwHEugUlWmEHLqKdlNnPraGiczDSiizmXfutvlo7zzTLCLBrY1VshS5+8jrsZ3igPNBmH56rxKFTBwXu+HaC39PkLUTM3bfbSprzEqZUyTUVtGwesl6sEdEENwlV0Jf2CcvPwlTnyjK1DqmSgp7kJLXOB9Cd+MJs1MeLc++zkeNsUC2/mFINcrUU53JjqjZslQYEpe5ux9/6m3S8b3V0R/ixxMwVUabmKuWW0PazV5KMlc/bS0sn0+63XNzjpgKkDZPphZ5E7P2y2LY0zlZryFlGUSme5u/R10OmLrRgqdK/r3LauW3289Mb6vXVTvPXGH3QaYCTrqoXxS+NYVRDeVOFWFfwzhebuCqRWUpK/uwL5UkCST6xC/FoV4u+uQprsLEim6KvnWIO0OeSydsIyDiAo9n/DXQWcM+V7bn1Hq6/1+3S1vkuCWaC/J+yDgO/Do/W/tRdf5Q3LRD1T82AWBghsOgGIGJRlvxbH4ygO5SzjSF4YyLbj1SN+OZurLAl0LBgJU0cg8V7XfAc23wkyuULlwrvrOdNVNIoKy+e3XfVU35t0eWc/YLTrnzpWMiu+pvHNVPOUXv2ubmQ9kSsMDoC1MskKi6HPqCcPHtH98hrG3sA4fP1myTau68qLID050pHQ01d74hrVXYVKLLJFqUv8nmTRJFdLFb1XcPZPR6X4XJSYDyD1bzt1DfAW/d1aL7trphn1+uPKTvRIlwNGFX3ETP4ie/idqTrMla95R6DE2jBIbZrgZVtBjDcalZmEwceLnOOQGn6edgHoR4EpP7CZD2J9dj/96c5+gNM9Q63Y3NDBdhhAD/7sy4uNj2+Wu75xMZcvqFZQ7ublRUObOp3OmL7N0cvvr39g77eEV9JrfB9rfvXc6Ppt0rhdmvTywXw8MIprwLfzy0BafBIwgWA7dbJWtIqEunFUeTXAOlaVejuoxhwE6dsPbudKhLnuWLbLUGvbG58sKuH4LEjNP1/LakYm4Fn6Q8247Cd40uWMzBRby+sW3O8f9MZb3Qzb7u6jWQvelolNeqsYUk0TL4oLrTAz8pFSo67akeJExQIIt/YZQ225ha6//wf9od6Y",
    "plan_agent.py": "eNq1XFtz28iVftev6NCVMmBTNEnJioYaToqRaVspS3JEeVwpFgsBwSYFEwRoAKSkmXKe1/sH5jWP46RqX1Nb82b+k/0l+53TjTvly8yuyrYA9O30uZ/Tp12r1XZ6z/pnl2Lw18Fl/3S3LTpi8w/hXNlvV1Is7aguXE+8fNE7O3l60hcT24+Ed99d2DPXt2M38MVkJRbBZPOzJ9WjP5Fi7U42HwJhXC8eLT3b92XYWN6aQsbiTYBpPVu0QilshybYoVHS9Ty5CgX1boiXdiQwyyy0l1eY9b68WXpBqJZbtwChV/zkSj+mAZj37PzV9/3eq8vNe6GA3HyQwvAC16cOjhS0rTf3bVesNx+c1Q5vCJ99dy3tlVkX2MZ4lcx2fHFyefKXV33xiD/c33zAKF8uaL2P/xJTNasaKz7+Ipbh5sPEjeu0U2xrQMCcnVygUyTQtIoiN8ajsfkQ23EEWFwR+PEODXMwcvNBrHxZmpYhPO5fXvZBjjB2JfYfBatoF2ACTRgZrNYyjEEpkEs4oMWHyGyIFzKli4NFXH9FqLSXy1D6E2Bf+kQM248bOyd+LMOp7dDUIIoxC91JXUSAUVq+vZCElbX0IssJFktPxhKt9tp2PXvsSUuRMTLF7nfCUC8WjZ/YsW0KA2MWUvSZXjI0Gzs18Jy7WAZhLAIAHd3in9hdyOSjv1osbwW24i+TT3EQOlc7z/sXfdHFoMbSjq8aEzck4Izk3R5H9NuwrKkLsCzTPKLJuXHYaY4wdEhT1NMZ3oAtDPWpdr2omaOdaRgshNSwCr067TrwQfNIbP25J4LxG4nWruoaAamSKAHGvyFyLwOwtieZWkTDCPzk2P7EnRAXEF96rmPu7OxM5JSf51bSLCNjVkfnZfebA7OzQ8sBf8e9sycnT3qXA/GkL45fnBwTmsPQnYGDQO/2gQicuFEXY5DIZ+57I1c3AvIC3mD5sX0HUgNm89woloCcwNs/hKS/GoiX/csTzK13FcmVp1h+9zsGAEIaE2c+EuNgFUMA8eAFDhaxF7YbQQoCIIFWirDvadz8BgT29w4Az3LvMVjzEuzLC6oV6iKUszBYLWkI8CMu//qyTztakU7AWJu0g/i78IJZ2xRHDERMOIWm4t8rH0ST4E3bBWcTQpX+im+XzLwRP0XionfRH5AUjINwIgwM0ztY+S71B1Njm61vmmIJ1lC0CQMHSsidAAHUJzLrvH62Hnrdx0yLzT8XakUgAFiAfGERgEPC7oEwBolnhkyCCrK4cGUYkbIY2559a88kNiiA3KUHHP0NhP8bc0TUILGhhYGzlNOYKYlBFvaN5XdbTfHggTgwuR9jlHr++I7fp0EoAgHVQhN0dHMjkjGYzl55sWEEw+YI6t6PDX/ZIFwbmBZfW/jaMvFTF8OR2YAKgQYxArUM9ghJ6UIlhdALhpq2LubytuvZi/HEFnEHClj6umkYjzDPbjJzBlbWbJosVKGtcMvEgnJbuYppGDPoHfH6EXBoT2ibWGeYTjJ01dQuTY3t2OGMIDSGhiseiGbjoHXY3Ns3xe9FK+sX2v5MlmA1sfm560+6NehDqDtoCR4QMypp8wq94CN0JB2DAU3+dH0FPSRoOjSa4lsSYwG5xt9bXkXBTquI7zC2OK2Wdj23eIiZ0/7DeRkI4U5FdcrREabFyBZPFcp4Ffo027ADULS2g5GQnhXJt4m6G8i3r4PQm5xSg+qjjXjS41loT16qT3WYptP0eRkEHuQ1vLVAfzXUCWYLbFuPPA5mp/Yyt3DScNaD+nM8O4oEzdabgbMVAkgnWpbru7FlGZH0plCH8yWQHUk56TYT6sBZmNTq4nl3H5ZHrl1Hds8gH4Amji1Yr/C2u9dWb1B4y6h7sK/evLDbkrt4WchFt93cP6yLa9fXnVrftNlCtbpPbS/CbJEXXFv2FPayi47TwFlFSdO1hY12Ww2A9ODB/DpHP4K6AaDAHQo00EwYNWc1sWtEODZvDXptuJGVWlYDHhNmFjVnuaqRMaNpQn+GacDRYNZJsGhc8K8B2WqDMGKmqzrEjWpqL7AnhsIaYLSgrNlz6iaAAXjpzq7iyAp871ZtyCyCv8BkBc7AfMPapDYicgxrl8nDC/XQmMnYqMWzmBfzQBk1adZ2BT61otBBU+2HZa3UQl+XNzWtbD/x44Ru3E1G0ktuKTgjQZg28luuFbrZXqSt/Ja2mo04MBLslDHB6LSUezRx4S/Rvhe07ygO8a7xd5R0l2vbMyqTXNkRRkcx8Er6YN7AhHANrl24MLwRa9KomSzmcxJzvYi5xf2IN/+EDX9yMrjsnR33Re+V+NOrS3ZGI4iRFImTm/iDf6zAIu2wnrzAAGBU+uqsSL3naBNuIaYak2sg2WPYkjXJ10Wc0T97AneBXi42//l9HxZXOkKNplgggYt2vaRdayDgFcDVg9RFhtkRYH6JKWAeLBJ7y1CrirvcMyxPGsyduuB7CjMAG1yxEA42xwNkVXrHlyfnZ4MiZp5j5881HY/JyCloLsWueC6+/Ocee+DyBqQATaD0ukJNAY+guOCS2oycfiX1QApOdLtC6TilEzKtayo+XZDue65VT3FSaDmaNVWEOS2YMGmwpC5EtKN0DDQjKRB+2Mm2cnKG4Ori5PziZNC/EAZFelKsIgACx/Jgn4OQJM6I3SiCS0NOGnnEhOe9to774Bd3xean1gEcwlMJT1H7ro/Zd2XXkfrjdQq6udi1oGgO4VWUg8Yg4bJoh7u7C3gxEUFBs2YxZbCCCzazPQ6fyJQacXj4B/G0Re7j42YKTrOxd0BDKY5pNh5/g+d99XzYUlMeCW/6uK2GNbEOerWSHolfqsDa/ARMzFabn+H/wcNb3dTF/uYn5brAnYSw2ePNf8FfXDt7e+TuAYuI54p082MiQDPRI5JUMP4tdlpzqEZPkZxF2v8YjsRX/pCghg7HmzKJM+HBwgeHd/owabwRL3uDweb9QBgzO6RAU9hQMz9H5IE/6532xfn3/YsS+41tf56AuHKcXwviPaGjZg7cwT1pyJ5F1iW9Br+jq10OI+H0OfhyXeT1We7T1oWPexeIvk+hvnov+sRfChCWZQDy/NUTmNXo7WrzwQTlxUxGxO1gDNLAYEq7RFhyKBKMpP5E8sH1HUTd8lq/+9bUjhJt7K09Jbeki9QkufFN9YdBpnTO5j9AkBZHVCAVnHxMk35vi8ilsI2gFkaIYJBzDpRLYCZgdzubnTgB6qK4jznZQI4wEuRCU1lVBF9bE+WjtxqPt+F3cA78ciz7rD+gJ2PqbX6GnNQ59mFdEEuPMxkcE5KQy+lUxiTcBHtdXJ4jcNWEgbadBv5kFZU9sEWgJET8lp97MF3vXw0GJwTpRBZyMxTKvgmAUo7/KfExUWE/rcyJLBmWmCF1NgFZ9vwroCoq5hc98f3J8eX5yQVx7Dal/GZFkX8mvzrfpKWpZJcqNIWrXjIZKkeU01h+7jm2yALTh0byyQWR9Aw5zfn0/BiENNp/0CmMqestIqUmO6QoNZRwy1vQwRwaM3/8HR0cO5KIPSlP58A9JjOP+B5b/XPvtHcygMsXcOAtiDvA/LbvwBmZIVoHQ+VggDL3pVLWocQS8BFCzszRAuJw89MhvkcxGb6IXCvFikmiTujVjxLc7inhyfWt0/xrF0Yz9T+KOZe8LWmb4qKvmKxDwCM0oiTHfe5aSZOYzHEKLzq/KOSKMyQZlBm1n/WenW3eC+xGTO11AFRAHWRojlYupURKHGspzap+/wY5KurVi/7Z0/OL4837fkp+kvNl4E4i3tQ48FeRzsw6EDcJPRFSapPN7CMSszewzRLsfQtVXASaQzUAzb+PEoXAWimCB5uYBz8AmnJ8i3WdLRoNnSxqYm1ylAly+m0ni109O9Zx6ywXE5Kfr6MzP2A31si15sJ1iqmNJPDwY0ONQfAAnzkKQsr+6Hg3jVcQoPgzwxwSwCPTHDZz4FB+V0OTZXmTBK9O6+KX7y50GNO5m37Ui7OJmuNtFjcw0Eu4+b2zS84zuov0nGAiQ98Fz7OBhOu4IDU0dX3byyXYSKX7gZ+RT8e7dmSHoX2rk9MTSrR18Rmxw6H5FTx3KMbwIFOxIbev0XpMjLYgLU35w82/D4XBCVPHdq7IyYkpYQkLSWGI51E8k4EHB51xSNGKUTs7v7Revuj9tf+EIlhyiSxyiRDIIaKZNfwJMPa7Lqwvv0ZX9lLSu0HZiIP9EgekU5Pzn83FOaScMwND7Qcxs2cn9XwQIjlBODGyfvWct0OwndfMwmpfrOiLLpRm02Ya85GB7A+gB0EqO165cbbMxOXsBaUYtbX4Lm87zK9TH+Caj/9Sc378hfRC8eDEoNgXXgA7QF4a/RYpl6mGFKllnGYtjIoCtkXZcHUozmanxd6Nd12voHv/SIlQ1TqzZz7lmimNG8siIZQGGpPc0+aYdfiNhInfisRnIJOvRPq5hJiYxFe5jhyGUh46r0C7ibFi2wTg4MuWWRDgdCpZmKKyrDQnqM3UZ5VJEx2aJJSLA0oogSb6JBBpknMbFES3IR0AdcVBkZqUgd5K6rski0xGw55MjIJgEUPn58QfUzx6JA63NDV1k2mWEU3E7lRsiVFA1nD3sDNKiZ10Mz9hlj5l5PKisE0gO7/Orp+d0Bls4iar88q19N0wPYN1YSuyU8+dbUSrEGA7+SmkLHBQam95hhKW9QIzCbUah9oQ1iismrphRAksWsq8Q8gLm4TzRVkD9rbI71R+IZSMO3XZ/yeQWMRTjaSFfmtK07lKac07T4Q3hc2ERmd7YYCr9B+zQalDA3Hgnpm177bMrUtg4+Q5YITZuTOtutUvEXGWS/b9xnTlc87E9tAnXNie+4P8lIuS7upzrgqEozH1AhCQ8rNwQe7aSoH8Vhxk4DkYPCzRN+kE8sZzOgo2+aEMpnNVhW8EoHbbjw86o6qtDOVCLsYytCCGpG9/law8VNlJygsFKzpsvpILOg3T9oHgodPClUpZLoIqGGncpR6OsqxLXg0olyCvBbZkQvL5hWZZKq/ggSVIbSw9StBjgprZSVKXeedh53Pi/EVWtWrybRHDXEUuu5SkR3Txx30EL9AvnitXBbjtunCSxC0tdCQ4qZZnSe5CslEAj7OsEKW6AA/CWa6zeja3qqq1O+Ht0GA+a4Sfasm3K1s77mgf7kL1743ocIwsQKe4rYkb6RoNN6Z0TJpKo3wCnSZLU8V+SZKO4rVnvYsnm/fb1BsfBOqFTfEd3M1OliIsKEvqsF2v0p44kNnWkswRh6lJAVorNlgdgEarBZ92rtVhxjpN6xM0JgtAAdw03dnJZycT53wchFzT00Fki1BhDIpnp8JrSrWmyco7/QFaOI+RZbA0mlU8sNORx0ZGzBTE0VaGiGP2N1IWJ+z8vtAEV6Sp54bXChfal7kjIie+qS53DMNf4bjPcNtMHyHm6DilkxROgzEg6TEy1hyBGMMZNAaCRN2jVepxpOpTdGu71Fo+zGi4i4b2+WmthzRmf5SwaEzE3lcANkuAlMQDUyTefFadpuJHfEpOV3a+0sQXHOec7stokw7DDmYNJ1jCdqoUEcJY7TpP0myJkfo5FHIoB8HchpOc1brbVDGYWQ2ALvuoVAfBgFEsbKhYuHVgmrT9A8IkB/MVBkjNQTCOZLiWeV04M4+4iKEUShyVctNkHvhkilVeekyH/s/UcqXuD0VLxR8cdFD5C4X8EOHNf1OKKdp9I+vQ5vdtF0+lPI5wHTdP2GxnHfEb87iPuXZqQFHky/OTs8uBeHHyJ6oQQjgJcpLBCydMS03q3GkXZaDJTEdLyrpxHs+j8qDdlU+u3yrGuFsZFrVKGN15dk9xAnyTQ6q1OaDSNU1w/gXBYVOEIJs73lqmiixuLKqWIdG5terixuIqlojnp+YmJRUw5nFdtE1z9InIN8cv9KpqhL4QjSrybSVpZh3haipiSp9IWE2HUgsVLcqYEjLlYrMC3uZSUsJxGBSrmGgfxrwuAigi0i8q4uK3lnozCwxKodB2ha0iIljKvc62xfgT+lKFFPVqMpKa4lvBNUzfIq7c56iMOhYW0FTkCWg2vFYFce3CoaGNDHXpjGFbiZoiGGwrow6goHXx6Vtx1lPlRBZFCwdKv+oJDkD1G0wCTlFzGJZTF5aPz4pTSE0oLLKZqLDGb0kt9c5OTrmwhTMWDOOctGo2QJuluVKmKjuSlcbRKAi+4icIJ59Offzl8zDm4vf0JOrTnuU9Cs9UPlInbGBcfK6jTLKaxfO9PxYdojhRlnwWli+nK4IB2tLZ3IhUKdtdcpb4ucXPLP6ljE0h7ZNRKAY6uOBtpkrGZmCcdVEJD+ejhhvLRWSwMSgg4d2RTqTntDx/MOZVLlAp/t/RyR1MYjIOK3cw6YN8nwIk3DGB4J123HXK3mZ1uaaqR+8+LETg+ytTLKEZdBz0VlWIA16pSk0LVSG0xnxCiwyNirSAu/FsfrXUGNvEBo8HSsvOhkpmKG68tUixtA5Y2/LT5wWsGG37aJZWkW+oiGY+wVhiEtP8Us3LRZdcKs8cTPEQ1Y/bbiwMsBBYFhpEV+WqY1KyuDC08J2jAL06YGwvd06qae9bWhcyIYez+YgTkkRUgpRo0Gw0TbFbON99IIwWPmFvjzCDEm2Dyn+TgmQst9+GI5ydUNHJHb8l2Xj4wthS9NkkbZogKxqvzpdYKrrjkJyoZc4bRdwUe+k6XlVznJyyHeny3uT8uRjmsEwYjlUtniVac+ksswj18K1UVNM9MMbpDP0HmiiyuDP6Rtz3OkKTQ0elPzpWOk91mncFoJgl1YJbWLKzLTZySDog5sl1gWpgaOnk/bYtHkE0KQuzRZIqE4U0UbvBPiQDGVmm2qqyC61GmwmvNvmDDhEaBxpKChKC5GM1+fwZpn1Iyz/QKQClvCheSH1ZpV81HE3FxrpMXKXOYf5kHCenyqmfqqvsYlWFQP5h7o5KdEvnxXQU9jaHWJ2kUYkoKAJnbqTJm+Hu41a7MzJTCeDODJbO2Khimspg+p6N4l7pqMyI0Eh3IRv0Ty7yzKX/pM/XO7YdTU7tzPBRHUoml5nP/225wCWX8UlDLWiZVpsbqtmlQU3hXl8+Ou5d9l9dCCpCwpC0cl75klnJv5LmArQGZXjMeqHqIE1ZebZvqJCY60Wjug5w63wzwmCtgrEkNHWhS5mA0npajVtXDFNXrEYFeMaPNUJQrSMuw5V8R6RghDEVfnxnlqtoyh4U+YtpFbUC/ihfDJT4C0a2IQg7Jla1mLw2m5FKVq+Qobm9hU6gPIOhRRobXxoOl/tT4LBnmqlC003NrIk5zM6OVLZlz9iYlTabqxDKmVcGhc3rbWpeb7R5/YKFaBbbrN9hNX5znKgrLugGmDIHXBNCxqJSfKGu+qQ6ohgLsNeWaMVOop7Nz+hqrfQo7HhXzcblCxSMYMIswPggamkCEk3Znylju/lFdExreZW70U2sINW6XWzePzs5P9N2vqOvxcjUkIv/ef+P9MIQ3Tcbe+XKcl0a9LCgkOBHxGkZUYFtU11YPfWZpSyvETIrnmErWdopFVjgIyT2BuKaQ1vtNn0H7t59GZLMrMyicDag6y2UGtxSUHFP9F6+vOifPbnoi+O++Mur+/3BJZfm9C/Ek97ZgL7+uf+KIqLKSQE70GAf8q5NKuqiPMpKe4TlE6fTnNd/JOgSHD6c8m04ayzjayl941RVotfz4VpesHjUpw5Nfcz4tFOKk9ShAo8lh3u75jNHR0IF36OKM/OmzrH+bM6CIf3VQoZ86YHrmKsOjX8T896eskhwr+EbykSpI1hshN9gpzj/zLMoUmpCbXORUicB85LSpUV0IrGTQI74KpT2vDLc1s7TbD7kOJAPd9vwRPChDcWqH1ujskjyhSRsGx/2iucS1ZT1MpEKkPEpwkB1vFEnQHN2R2W2y5gmNxS7k8t01fIhAo9gu5hlmsGDBunrQn+zM9KXuOTSc/nsIkrKfyIO63XxrLpxRwozKeLdKR1kyKVefZvuUwWdRu4NzCWXdGi3d0duPq2xLKfffQsOW7dUkwkT1Ca9TJOX2tQGP/6LDiKAcroClD1x+Wzgq/vOH38p18nbdFOacx0IyHbE/8PPPSq+oltU0Kr0r/SpOiY9RqTA5/uL3gldDOUr3l5yNTghiSzXAGYVbWM7dq60VnudE71Z6o7qE1V4DsoxHWZPMjl14Iqpa46/r/WBxUjZwmtqeU1q+I7j4p28TBVWHGL+VnH+3Ix3TqiksTQTM7qEbJpfPV/ZxsyUHGY4TLlOYVHxnrqwppkNnk/+ACy7uN0/E38+f0XldRQ9X56/QhjLF+LpRouE+10uSk7uYNPV9s2HMWWxHrJAbimcH5BKWG85qCMyJeKdnuCslWhWVMggPQ7NJLOjcZH2ZYMwVJkjv1647kP3uuEm5i79JDP7hWtS8Syu5dLa6a0hEu7KJSFyxqseIt18STV47i6M4gU8u4tGb2IvXht0rTV3N6ZyoSsObajC7PN4mk5jI3p04IPoO39J+aJqHHPFA6lQFXFNutl9wa7QtwQJn7lrgjBplmI9y7GX9tj13PjWMHV6+tAsnplu/t1OS2rpko54FjzSrLH7nTg/P6VDtfZjzS7F5MugqHcpXF1ol4ZuQvHV48rx7oAcsTIrDYpcMy+q5+tYn42oKs+h7rhLF1j1HV01yyjBHqMN69DIR1361aCjZrPiOVjZ3d70CLY1UlkXiE16EsimlT9tcSdeV72S1DPRd4z1qY5zFYAujJMBRZ3uD5Luii67gPCOwhu3cCyUO73hSYZvRikisN/XiYmnhqHbcfF9Pqr6AlrnJB6f1tqvqx055h9PYaQprdTNbjlGfLe86DQXmB7i0fhBhkmd0JHw4Jk782s7nBi5q2UNspdG7kLkws6RqUQipf5yFYRk1VWux+x83url7mbw/3Pg8IU/dwEtRwqwWF7D/90H3bVUt/+1AYQ3AsmKSXSKRYjLBDkM1LBKsdQH4g4mubPzdkVitFdjVk8S7uQBcoJ2aS6wQMJSlHHbSmI4QAlbtHO34KO4xCSDVqqgiIiR5blziTgaotsadsjBHFWMIsGkZibllOyksEjVKKoTcwocOW+mLtyRCXRzfsbOV7KkSvnXxaA7aNHla74I3FI5xUIyaYFoji8Gm2nKsC7+dNXdsnOaB9DdOQ3aLDs3z5YC6v8zkbjbkH3ytuuvvP6qL0GWbVrp6nImlDnb1MkbJkT/8a1FdfPSMD8jqSrghbSlhopTLMVbSkbk+mAWhnEVh8l/ZULnSWHAnq2qj8pcq0VkLRF4qx0x7XJaV/tjrWazmZxf6fzDI7ZkKuNAGe2d/wUWVfEj",
    "cogmap.py": "eNq9V12P28YVfdevuNUCXSqhGcmBk1SpDDjOelPAH4Vj90UQFiNqRI1FznDJIe2NEaCvfe8P6KO3BfIe5C36J/0lPXeGoqiPbY0W6AK2hpy5d+4999wzw36///jRy1cX9Ozi+atHTy9oISk9F4nUlgK12NxK+vUftFJ5bmKR5Xj6hT6jWBRWUmwSrayqJRu9MmkmdEi1SL1ZLgp4qkqJH0n3H5CJbURjdhefy9JSpSVlm9vMqEKGPX769hu4H2BNUoh8xW5L2txaYUt6ufnLxdPvqd7cxlVJC6FL9qqxuahIasRSFSUF373+ljJRXlebW7hJVaLhAvGa1GgeXleK4pXQLr9KUy4r2vyN8kLCRvIUfnq5KGkCoyy3sipCmouiQDTnm1sti0TJQehCS5AFfmRZipvNbRMVXMRwLK1DEuZUYK5KkQUFLhkqK1ULbUMqlARil4+eXdCLP128HES9i5SzkvTGVIxgycGlgvJUxAigoswsNh/SZqixw5gkm7x4+QfU74LXamPdPGy0WqpYcA4UzI2uGIpMpPgtK1eVEYrjshjQP//81x7RPf+I3LWuHFoIZimUdbFSkG3+nklfkwEZvyLbfMAiBNomgqjcNh2HoMQbLGmw4tpsbhPs7sMJcoC03NwWTJZzpd32QPmPT19/z2Xyq1zxCrP5yZKoZewIYLJ54fgHsErQdsuNcw+12fxMKvUhIhqiQL7LU1MIq4x2FHVVhiFCWtrh7+j+F46oIY1GX1GcqtiDdR+cLyXTgolqK4XNC2nBOu3IXqkSz5kqZUiP6N5D+ob/a0ZRFA0cFlhvibl/AIeKFfb3JeqUhgngKonXTHALdmEPTY9fPH/+mmoJygMdnyqjI9AKsSlkl55vRCbUjqUeBTQhqLIsjLZq88Ej2ALvInTtixC5IcENWqBLY4caU9LohKwswAVuG1hmIlFabOe/c6EDqNoodlVG/X6/p7LcIIeVKFepmvewe8admXq/JTXzC8k92DzoKstviJ3lvV4vTlFlemySZyIfu0wWcklXVwo6dHUVlCG9vbJg6mIyjB7wgzYmn4z8ODGTz6Mhj3JhV5P7/m0ha1488O74r4zeoswLFdvA+2p8huS8eachwR37DMl5807R0nDnnA6+hiPIirTBoNeG6l+UgzF9zN8ZdqycwjVCN/YvuOOdBHdivkTM73/kTZ+0oxgitx2XFgZ4em60dKEZzhJrTNp5qzEedvZ/gi2Z8FxfRxKvkA3lgtywvklWq4KZ6Ok56ISVFfCo8+gHWZgyAPpzY1KHTRafmtlBZeY4OmrJRRWY/A+QncGmeufj5B6KYxAeoJXcgcfnAPjPR8Wvv7ShqiW6B33CSJApSETlSuC8+82E5n445oZHv7cmCyQg3IKuFw9tx5N74V1MhzNev2ifxm0lWiR2kw4mX5+j2ZGfdfXqwK3p0wmNtvXFmA2qLBgN6CENB/TbzovfT+jzdovd0uHh0mGzdD9JTQ8n9MV4W2K/4Wd+gobRl22Jvf/OzK7Ea3nD5U06zbfyyeLAFTdBgnPW3uRygjdK269CFDC/mbwqKnkQTVbsAJ5MaNUBeDXl6ZDGsz2svFl80mzkzMahW7Fv5imw1bAoWzwIVpE18xtIbTAYRCv5bqFYebtdX6tSWc50HTpVriADIdRXZHLCLNnn9lnTXcJaiazRgeVOzXMDWs9TjIIDYSfoA44V6LY9AMe3fsPIcUcL1h3mQCum65mvl7ZRAplCtKj9pzTqunNRszPcMzzFhV7Q2j3iZIL88A5PvC+3uLXWBgLCG1xGUEEAI3At4l3e/7iLeImOSdbsaovU2Bl2TRLYONy6wopzb7HFGP/0O3u3ZJx1LmWL80o3l5QxDnycxu4sPLiq4a7Tv3zR7yKx9vlecr6XyHeauJyx8S4sPhRcUB+l+Gcnjnp/IkPE2rvayXO/e363QeJaW/MBsB47uHAOXPPJxudrgIB3oL9d8X3meteG/Fdh7XWUmzyVS6Zzdw7pV6xka1d9oW+CupU8rmDdYON4VLkKR/gyqFyT7G+z25/DnVazRiQRNEewe3lk1fTidsVo5lmxt87TKTwZkLIyQzzH4SC7+ojiNQfFHNg9Vc2w4b6D279pmNE7VWUXb81U4UCSNTT4OhJ5LvUiqAeHSsP7N3m1rHIXxY+m1Rn3Ewgk3lRgT8ZfPsyWO78Udh8Fpzq3EQbg9zXN3e2idxJr17MnEXboTjyWuPQ2bXMP165pPzH9/TrLdLt8fbiWr2GnV3dKd2jkbnL9Ge7i7hGlwMMnfOSkJhnlwU79ala/PSV1KDi+p2mwT5CW8y7tO5ie4JJz5WB0srA+aqlkeBQ6H0y4gHBAbD6MnCK72NlJJ/+GL/N98bmaS/tWSu3YAgfr0V2UObvzO2M9ZPqsR9DHqf+EDRuChPxpM6PAYVAq2n657cnksHsZgpf2IOow/Fiyhic1a/g/ixYzaXTcmPxhNqHp7N9L0355YLPt22YePfFxmgXT6Xh8bzT7P4nVnQr1X6jRjmCVdu3UKNHh2veQnZMZrfcyahNw5e79C5bah/A=",
    "wm/model.py": "eNqlWU1vG8cZvvNXTC2gmo1Wa5GOFEsCg8oy5biQKEFS0oMgLIa7Q3Ls/fLOLiXxUPSYXAoUKZBrbo3QX9D2FiI/pP0lfd6ZJbkUKVtpCcvcnXnn/f4cPnv2rHFy+nry7XGHvf6anZx2X3fY7ztnByxLy5xFUrN3srxlB+eHjA9yFdHKzueTH3Y+d1lzhwVpGcky1w77z5/+CnAW/PI9zoUl+/nvrDe5xz/2879YkYtEZ6nG216DsV++VxEznz1mkTKeJnJzmBZA6rDNL1kkCpkUmmUiZ4HQAAiJ4OQH2t83GDYP374C220Wp3cySSS+e4pQdU4OHBZKFq0bQpzkcTzQzfLJfaiCAhyDckWCFw7bYAeHl29Pu4yrkG153hcsVP1+SfyCgSItC4AEeJIkWhCpoM4kLzaaDjs7n3z3+u3lBTiFsCAudSYCWQG5DVb/ZEIzmbBM3cpIk0BBmozSqCxUirOhwqHJPVS9wcqEBUMZq4QNorQnII7s9yU0E6UqKYQCOClRjBROuMATZySfCwOCXW3k1mkSAhmpe/KjkUjlpKCruQrcRWGspgp9TWJawWFQSGM4ZiMBpkQykDlZl1uHcJgspg7BdKlGIsGx/UXBpfEgLfMRiBE30frBm073kvGgzFWqSQqw+aG0gCKwGumngKZF/IF0qqXL0sk/DEkgiaUuc0n+CicIJz9F0mucSeK6bb2Zz/ypEoyNtHWiQPVwKknzWEQKBidnAONqE2qG2nIZE6N8JHIlEiiAFFkhI9BKtdyqw53KD7U/Q2ipmKzAijQPhq798pKEwfhJMn/3+mVi5IR1sXXUaHQPwPjLudbWZoowzsm3sH3euehcuuydiIXScBNpo0wRHBNZlistnUajEURCa3YuNQelkzQEf86eMUoo+8z3VaIK3+faZYgweF67WW3TR5eZzLnjzcCcfaa9oAn6wHYIp22FFJn49wIuJ8JQJYM2sLjWicELvdlTrY+fahqoxAK9ydMy68Iq/CXgnBm//TS/EXlI7N46eyyXRZkn7JYsAQL8yBvIqORAw4nN6futQ5+ZMjp3sq4M9unPWpX02DRNISxsRqI/Z7U6283Wy4/qcr7l6ULGVvILCR+HA4qIz5VFdCjlLqjLNYrqHH/N7WMFS3C7gMeXLnIVynbrsVONT4g9R0oIw0dx1iVBmreCnIlcxLKAzNbR4Z1hwpsGT6U39hkceqtVP96L0uC9XlIF+W8I3s23y1rzx8+dld4xeIpd19iA8VcuNEbpK0qTAQVYc3vGzy0YOfJgdB9G5wMP/hNnfAvMbzsewZPqIYgH08ZlIWkLNmoSh14/SkVRs3LlrFMZubU6nNN4L/Q299CzXFKhSvNPBy15mcuG7eZu66nOppJsMRTBQJP+YBXkKRuIQsY9C9WJe9JYmncPAPCYtSybx0oX/IpsA0xh5JBN8M1QwMj2LeNDLysVXddxUX1bxoTXY5VIkRO+FhxmaFH6hDGnpMt3gKcyZ1Uq3xyfvjo4ZvxIHZ84KCdVyUSlCIGU8niNLlX3ujaG1kOhBKyR9ryxzFMNbROodyPVYFg8tttTQjuGlXByT5UWqFE5c4Vqn8iqaqJornLZscuEa7qL94877xobk8uGLowFx8MRenWqc/Tcctite8f4JmVpUQZlMtfyKyyNPa3G8FSIEFDOsdFppMBpMgxlEPoL5UgFsj327MMczVCQzQ3Fqz2XbV2zL9tsa5+p8HaGMEkTwskBS6EgCtTMmheqPqp1gn6L0ngQX+EBiCox7FvzugqxedjtvHAWQLZWglyDiaa3NaMV9OFBJpJN++KXiR6W/X4keRBTAlnWMRRh9EsqnrUjthHdoMaDvT7oXqAPqHpT6ioCpanyNll26yzkDxNvVQ4MkBCuYGfDkafLmGLivZRZqOL2ZV5Kp9q7Jg+0iYECkYtKTscbKXlDdtpsGls158QoLHrRe+Q+E25jlfFphLpVfNUyhHH+ABtD8DiI+K0XS5Fwjgh9gWLpBcMyeU9vmxQKJAhw81tEIKdcoYNlTgy/wxXryykQsUKJ79MJGjp+/fboaPLdead72IG+qSuXZi6I1m1XxLh4B3y0LDCrTFs909lp6nOLemJNe7+u+K+xq7FfuOzf//yLj97YdMRROqAe0kazbRCnPfDGcgvMaW6p9v//TqHMPtInUIKk5EUlYXWDcFmNYpK6CgKrWoXWwwONp3QGdWy7VY+yAlmd/2A4WEi3aKps5Wmu6AaDNHoEeOcB9CMJdZyZVmBGv5zlJxOKpM4HoTnObOy5/3fhv16IzpnrQwG8dGzedK2MeIeHdr7xX73tgsGIKp+tbUC7beLqavca/9d2mthqtRxbbDCiovRh6kOtebcuFOul5cDM3i6NKBhQJvejyX1inFWztFTPkZ8dInl2egGSV9swnsu+QG2+Zk9qiIM0z2j2zNJq5kCgnWCETnvvMJQJcv3nAonxJ71Yiojk+ekbLLRa7Nd81jC8pQODcCRzTcNeD6UbyNGyOY0uXQW0XsybfJowv5IifHK0r7EL8P+cXZ5+ffhV5xxPh6fnZxf4JobPJ98i45dIKDTeP8gJXCYafUEi7IypRiqa3A/U5F7/rxFP3o8mcMH7q+h+ueBX2oujlSmhapte4lBrG7bFf0sZoYIx+92O83gUzWctQ48b5h6JnFm1b9Zmrj+keRTCDDJ6Wkv7SeWQakJ6kHdU22mmC81GZtU2b6Gny8j8Zp0qgF2TIzo5c5SwrtViUCygxTsCX4Q+GshC+oSbG+JebcFZrMYZ1WF7NJtOQ5pDlxnU96FUudT+IAdOfiQiM67Tyd9NGyizx+dGkbEg/cTtLW93d6emIUPLZR/mZZ/4qpN0V7ABPj54cRn5PHY8JFKfZ+j0ChEMCV5E2VC0m2yTxRVfxEKUFE+ermYV3Nbj2Y2cvbhza5cu83u+h+nyyIvEncx9guVGBD5YSLotm3fBMLZD11mZkef8Z9YrjAxPa7ftRei6GFCZ32N8PAt7d7kPmC9V90AzecamE4RZwP8+QsW8kjYhzlTpdg6kwKqPAmjTh3SxFJkjxo35rLLtTxVVrVSgNYOlWj+QFi/Q2o1vULXRKLtMjtpdFDtaxSNm8ppz1TmfLd6oYsge+OkeG6OfHBc10dzpU23GN7KPg9XCrrZBpXAqjcvNHRCZ+W56t2cdzbaGM6LvZCbMBKDjNC2GftT0jWZIZ+OixlzfTEgrHAwRq4cik5zaWrgaVD+C0xLOnO6YKFLGfSSDEKOVY5tpp3aBt3yn+M3bw3M52ASWOf2Pmtq2TlAm+w1a9uULhuyGtg0nAK1YcFC/5m+2kfExJPOm3HzhTAcnyLS95TwWArIkpdrb6srbTTBkkGZyn8t8xkLkB0Ojk55KRH7nB5gqtQ/oPM3ufPIa30YIN5IOB3RRrX07VLezG6eOyejhyFvAwbFKDry6I4M7lcbw7Wfob+SzZXTcPlRTzC4ewMSSvX5NV6JL+wtBNPnRjIaTv01vxz8eLQsNdhbQQD1kX9IYXVCEcCz9li6WB7QIDmlaXLa5Sf8UGdQc4OBzOmhhId/suF1ZMP8cR5FSzJoQ2WBb3jYwkWtvTHMEacsYdsOqkeohnTCZlI61LVf0CCPgbLVArm2sXL0TEsc4+HwB6FxIUC3069Oi6pt8RAIgfkJkKaY0ykbBKFntrao1ktpd+lUAT1VTzJJ1eYu+WZqfENbNbw6d7uX5weTP3c6J+e3B/MaFfjmBM896tsXLjj9a8ymNHo/LkW3fFwHBLmA9kdw9tK755QNIcA4Q1/sstTl1xMdmgULcPDhLxyIfkfS0iEqJKTtEAKOsvS2jzejmZwqCEWB6gxP59v52KVGmNdirLK4RmK44pIAstvIzZArJ0soRP2NbKwSjdn4VKavpQiTDOVU04NdOjaZ5X8JpHZn+37CeYxyXFLhRSbZR0TUujOEvRD2BOQlm5pJ4JlIjOjBbpOtZu4jT81VCtTTjgQG6J2/8F4gYZuM=",
    "wm/model_seq.py": "eNrFfU1z20p29l6/oiPVxIBF0STtq/FQw8nIMn1HNzKl6MOTRKWwIAAiYYEAjA9aUqWm3uV73+2bqlvZZZGqXE2qsslydsPKNj8ivyTPOd0AGiAoyZ6ZRCWLINDoPn2++nx1e319fe394dvF/z0Yirdn4v3h6O1QfNh/u/j+UBjW3LXF+8X37w/3j4emiMIsFrvHe+K//s8/CHtqfcpcEVlx6rnCTVKRBW7+prO4t8MsWtzjSSCSxT2aBrabCMcVp8KbWRNcbwrLTr0wSFprdjibucJ3xeXiHr/C+LD13fBotyWieHHveHbqYmTbyhLLN0VfAiIfxXjtGXco0s1uS3i+mIdeKk4Pz06HJ+gyyYf7r+//WaQEgG+VEAk3XdOaKoBk27bY90XkZngpDGw8s7NZ5rsx+kCnAcDC59vd0Qn1qPBg/P5fxe7e6f7h6KXAjICHyLcwzO9/1xJ4hIa2j/mF2TxmQKIwTumpuTYQVhTFbuDIB/HixwnwAXD3huKjmxEe7TBI3ZvUbQkrE76Hm3hcosESs/DWDQJ+Kw2zhKeEd2/aa0L8x/8HauRPPyeexNv2q8UP26/E1i9wheYpIcCwrUSS6/Xih9ciujHp2vFmbpAAQ8IRO9zj1t7+G3DOQAzf7wqDaGZiLJ1qfZHGVpBchfGMvr45ONzb2ts9O9k9EAm+n/LY+aibIKakAQhD/GD7nk0jF9xGmP/o2uniPuH35RT5bTRLwmBNVH54iiZNzrdSN0i5Wc4xSebNLeBUMpQaI8XkvUuMPZAzlF+cGp8FmI/le2AkmnC6+G0KQBRyLabpvwrQPfJuXHCkRZ0HE/DO738njJTAQS9mSyzu54v7wJ0xZAbAfwHC2VM3fhHF4QRMkLSYgmKCZot74DXBW2gnkshKPctn+hAEk3TshzZu9AGoRhofwqBmRjSV9DwZnuGRkYAwguhs86DAzTMrJSyBAIwzEml+y5vNwksPeJhYMVDh3oBKDHUDyiWiwSC+u/gX4pWsEOu3L02GD89ja/FvcuKC2EZM/PCSEOoySZn9WJqvLAjgp+yZ69N9UiyxN6e3DoZnx4LkBeLeqoPgqv4wR3vxYwCRmS3uZyHkxBQxlBWoLhyavcQVSZfvTQJXsZ3GwCQ7U9dyxklsE3VB17uIyDhgQBXlCXB0SaolZEXAczKmXpKGsQfkm8AHXp16jnxXTid/7dniPrVSsT86HR6PhoQxDYLa3KCVCfoEyiMOSRRYHl01sGR1vJVAdcUEmvGqs/iBdQ19Z8BMEbBuYqFEX5FPysIqBqUeJRXdfPoivPyopr93vPs3hyOaxmh4KgzHsyZBmKQQ1LmXZGD3Xqe3vdXtbPVeEbEtJRBSiiEEPCRN2JNrxunu3q+G4t3B4dmQFNslCb8iK6j225lbQ0Ap9pAL6MQAwGPS77qYh3WF7+C033Ta35hydEUQMNzc8oB1sfgnqWDvk9DPJPYycbT/18MDwQteLudg8YzIVMc/hEUiWEpBYmIli91JjAVPzg1rgTh88x0hBwtbFCakYxIG1b2RWpXnJVRPpCNzdc0KgwQxCwjlWJ6qo9M6A+CYabyACGjxFFjdpPIVhWdaVlgKeWiMAJDnHi0QQarpsxYx/Cx0Fj/6zBS2h2d/dXa4r7opOMAOfeKAnPxghMMzFkLSaTMXZHGrpJeTJHIlObk+nJ2QiGGY1I1nXgD5IDEoAVAkJN4AnUBTRcm6WvcVs7LM5YBoCwXr3SZVb9jT0LuBADCdZp7obrfEJem4Aa/Q+btS0SVWbeCcZIRb5hbQia0dh+6ebY1cBjzyXQ+srTPZp8yTpkmdv6j99qsXL3svutsvaK0lDMWkO6mxm9QgyFe7rMAZhCaATsQHK/edytpT8ALmB4VxaUGNe1g+CTmA8Gj3RE41Jo2iZkckj0PfD6F6ieT773e/3R8Nj8XRwdnJPjB9wu8ZJ7cg32yr12JxjlzRJZn7mCXQ15hU2inMMWLCeWx5hTnWgoyDJiAImhCm1HxBvNp8T2CbHR0fHh3z5+L7t/tsX2H4fEWnHj1IPTqNLUgg2QZKubtYJm21erOiJ9kEekh1Mg1JWiAcXgCm2RHXZGEkSyiv6mNxiYV2S1qjQCq/A4GVUEhpZM2QsPCJSWw5HokcaSA3Ttg2S9leA21oESQ1EcfehO+BmWiBM9tr6zDNvRlZiAJCPs2vsaDY05b8aAeBwOBBUH5vX2UBa3AAh0fv1q7icEasAl5QPYxgWI+GLTH8MH6zP+LPo8MT+Xl8+O3a2hGm9XpHfIsPGGcvXoijHTE6xbdvxXP8a/7Z0MxHwjLTeW1tzfaBUjG8dQ0A9z50wGGmFGfHvRLjMZRAOh4bIJ4z6H2zbZainkChxobZLtqY5aO2DXDQ4YlLhjxZQtT9XhjMe45BIr39qiVewoGwHMcLJgMyttDg2+HBmSEvVVtq1+29bgl8JmkM5Td4VWm7Jh740cakPpyW6BXd9ExTPP7DaMPy1d2mv6+1GWLhkHM8soitoTANSWTY005gwAgdnWJIEzTptDs9cwfvpLF85bSwueNhQMopNhrvHli3eASwCQHoCFfoqyXWJ66frZNiTO3p+MqLk3RwGmeuNHy1G0BVzyyoib4/Q5kQMSdYV8geHbyz/AQUb5z6RBijFiiw/YrtFvnF4VWLbdlBxXSFpMyVjuAVRbkmBcpu0B6MYbxrh4E7noapMWmD+2aRgRl1vzHbfhhMiPrdbbMN1ppBDOkR2KRL02hf+aGVGiZdsB1s4B57LyCFa3SLqdJP7KZZHGDMTUUq70oBjfXbZVIYxUMTckD4IXNg7FuXaGIAQVB/cTLA5CXPQ+B5/Say64bDq63VpsNOHYfQxGRvpmz6FQ9YHZ5jrq86P9s2hTGQNl/kpp60MKQ/yw5luXQyNGZbHMVhZE2s3FaidXuWzQRZ8UySxAsaQNsUJ7tnp+LtUBwdkmVLy4ahr888sOWEEWnDZyXgypPTpsIzAxKgPyccUsBtH/8wqndnYaVNlM8LE0cCwR4d1mYYEw4vvYkbR2zQklqlgUfA8qSdeHfgAcjOpTfBja74+c9hxACrEb69a0N9EKGI9uoX7DO3/MwdbHUlM4CcaColE4KKxdYgJEMy3TlQOpi05YXZhvH1mXoilQOitN2bCJIMIomtLv0z27wUTTIYg0rXQaDEGMQRsmNmGE1D+iWUAKMZTkwM0wvcz+SxWJfFu9R1cgttdUP9GywjeMnoqU900lGf0GT9BxTYFC4a21CQeoscScydzPfqSi4hkHhSHGTgXr5yfYbT6xqT6LwPkG77yS34hxCV3PSTG76+EIMBq5WH2rSIkGYppvqo4OApKUL3syQFI95kdNXumcUXRat8qgl5oyQkoReQLVBaChB/OQoWJFqMaGZANxB3GbvWdUkyZhc8XtO0CNElXykPLz/+Cub20moJvn0kskDTIPHps6gouzNp9nfk62Sbn0PDealuULZE3cW5kKtglHmJFmRKhHr1aPdYOTyxi4FvKe6HUQGZnUnftPBsWrIT9jMoklSIY90YsL1g8LIHotJibpdK8lHDwHrAMECn6OxphoFdbwnBcDyftSDW9q8wEZZ6fKX1WDE69PlcPjCfHpZs+0tmxJqhcb3OILwPSHkuABktKi97al1p6atPbR22lLViY0k9R+9fvSqnoZG1nfQ2cs2LEnxW4S2xh7Fu2skU3sd5v3exo+SrstCSg17Rz6NVynmkMAS0kg4vZ3TFw2jQ9SSAZjt2eXCDVPheYULsCM+5kQpXa6GRdVZopTs3DhNjpIZEJwVwNwVwWJ3dmzHoO6ax0XULIGEQO1jZzVP6kG+CLMnY966xvNBNxvRAPuLZvOxpKI9CjGgA/BcYXNJxDG1udE1SyiP0dWGeo5+LmgZl5Kyg7o2iLjMYqxSpZEhzvD8c1eIgSmVCMHQGA+BRKNmDAOlcFPp0L/TDeJVGrXniBuDfAou2Ghx4tkGkg79TCTuXjrjRg+3KVhHcvSWvXAWi+jATJ6xw5fqYm0vAFVSdstc2sYxZ0JOUE8AHVP6Skw6Fv1J5Oj3Sma9eP01n2pfMqbNLxxJeS4T9lfqGnj5N14QPttT1m8sreNt92eMPUvf4eE1CfMmenI03cQntxepOflOa71V555V+R+s/Da+lBj0AQS04O728ETlMMzXQ6xVvOxIeR0LnwFfTm/eWhq+CV+mKoimDmjreJo4lOK5dN2r29uCHgHOMV+1Os+KePk1xT5WiJhaVnNusvMMpm5Rfr6unSpp3qhZ2qUsZh0x3I5yi2Q2W+YHkALhuM+tmHIWhDwTdEGdA+aAJvOOB5I5aE6KLbPJatnhda0Ao1hVYys3AFMZ0ybkrVLUyyb/Fb/nmLb85e11RO/Cb09qqJNuBcfSG79pkMMZRSEErg8xuuIru+AqqJ4zZnACkzR0BLV/W0ctec0dA+Jd1xPa03lGhe8HLxq3J7i1z7nPimya2o0gQZRrIbu5Xo7N5hDVXaIXGhuo5il2KtoXx04JGLXE6IEIfDB6OH5GkURzrVLcX3ZkysYazS5eVlUERModb201PT8ksLVpc0UJRa3IEjBzlrXRlVHS3Mqhz2pJv6WGd5PHXlsJB2qg+RXlUMEki88BLwAJ/opBQ3Ws9MC8a9WCukCUmcYPIJA0ZkJbatT+73mSarnqKNTIpe94Qe7uj3QNxApNhSL+nx4vvh1iUt0Vop22yecgxyRI3VpH+IutBUSW0H47eHg85LNwnF4tYlQK4bpLKlKC9uHd4aaf4sZPhJeOOrCcOMMB1F5dhNlnctzSQiBvIjgGTcMSdLQIgMbu7o+8TEFOEczcmWfKfUZhkRNFtYcBxSjLbXvyYx7Ip4EXjQoY8DrEm4i/MtvjbxX0ckrFBtQbwD9sappOxPZ008400Fx1TstfYt5/UbBI+qVmcPjgqOVKaUrn0r+sBlFNSxZFrpWPWUb5rzaGUT7n72J1Q1igeX2ZXV+h5fWYl18SX/vU52Z8t0b8Qv+CvhUXKpNDD9gYxKkbl7h0vLVdWWV5AmsVqcfnB9UOL60ZRMvL2jBIUFKYixnkGL9v6yHF+K7vRKhQqlRfGm9apjHgWyJh6JBw87nm73SZLVvxiIDqw9WHi2rf1h+XyvP3SbOnPutVnpbUHS7fAt4y6TCnVaqBzCvWbHOPfxHj8NfcSrjLfl14CAQJagBhK8zV29RPZ05Hs6SerOmJFqTEDVdro9gdBq+aBVX0A1bvF4SkAkPta6AGjykkzuVcTC9LMpCA8fcoW91qajVG3vNDR6mBYyu5h1yJnKkzMkOsDAykXQ14MDPpTa/xcGJ32N2iDCWqgmms1xhsn4ZXkPr6C/fwZ/+wlHiwZb/fk5PDs6GAf3GdEvhV4V54tQ7W09ubZJ1JpskvJdaNdk0oifE4OOs+KSqy8f3gW0WfZVIaMo9BzypIblaZWaOTSFargIspENr9mcqqag1VUw/P73wnyicmjIP5f/FMuaKZIPM6IcQ2Z8r8Kx0ZF1oIwFVMrsdI0Jtysjwnj62Y1KwyY6npkmxYwsiKVNxx9zt3hHQGLh9rTW+BzakSXP8FVpdekzYOR9T9W/G5UBSWXE6PC9MzzRU9DCZzkF7WqnauuL0rO0Z7Q14sVbCwDzA6XwhFnyrKDevabiWNdagUMwyDkCVTBGJ02QMByefGgR0FpVEsqNUlhTcUQYnU+J/4HP/2SEUGiYnQhyJUmhDgJYG2yxINOXUPUe49U/PY5LopoEFvwR/Kv2U6ymWHAU/nGLA39WnNolboKMJTU/FIpgxxBfXlfujdbXZg4NeVgu1+oADSf7q5cfVokKAO5sCWBw1ekDTbEnbZ+QLgsJXY78j35sEff8ZreNOECG7iVTsZJt/KJ4/qpJVMn6Wa3wMQbGKVA+V2JqUqo7465hyxbnrDiJjZZWUUqQS+WUxJo3BNewiiQ+TLcYLbo0FMCuPIUN6rhxZuCggzcc7Z9t8iVnkkvDTbBeT9/oC4uKrkPnwzU3Dbuc6++gRUpie0xvT6Y6UYrFW7Vh23lo4ov+tkQstqLbQ/K/vsZq2qtnAwqsag5G1DR2AoX7MZsBqhkKju9GduWPXUVWxGeH8rlHB3sjvbf7e/tshXa53LV5MXckukAivnv5QW1ZF0fHGDZAR/lpalUOOrbGV5Cgy7w7GkGOpS6hICSSnuK56hczWMDGyiQdYikUoy/bIkPJjd8nnO4bsD1ecnOR5XlbBFYWqqkjOqIgsoqsic5mKSV2MQpv+pxWTDfHqWZOn1SiMAdHFIyyIgVd3Lcnxt37cD9PJbWLOXJyKAFehvuas5QzmtV6TFq4rNXER8iV0FiijCvYvX8gbq42BHXczw+f4jnK+vcr2FBkwhgZP9qjJWWHK4x1cmMpcZrNT4jF2xHTPlN8gO74MlKv9fzthURZQzjXduXHt+0JX597vQpSOZQxq64NilNoDfi231uoy5NDvY8LKwa4W4q2CsEhFC91b34GnElVwsXHrxI4jW4hVhrCxkmSwqisiSv1/MlRS/lkvX99TwXTljJnaqAbkAhJ5609dJOu512Nr/b6orEhRvP675xolXYm62KYHB7tJ5zBRIJKo1ZAHfSEt9VNftKyWjg1bTTTzu4891qluUBCpYtOrN8PyzzshyOKnMRRv7G9fy8c4HfXGqruYnLMPSX0xwP5uJ+o6QFUNMIfUOCb6rxGm9fcAisKkWw9a4h3fOOSfJ050VGLlMEdM0stapStSMCkharHWSzMSl5iE8SlQH4tC9SHX0UhwwgCw5ZmcF0dUEK/XzC+ICLPSglRqVYwqZYEmmrIshm255mwTUZSTq16IfUcZVg1x2K9l6raG9ezHBSFDMAlAtZSqC9NOeX5g+8NK/imy0+ng+HJ52xE6Zjjr7AVyqK1Y0kMj5RYXxk/KX8+EDTBbqldmB+Wx3u1bjUqQ7NtgbtVKHllhFlhEr/0G1fYbkHNJMLNWfHy8hvdxX2e8C+aT5kThAErSUxWbXWV1vnYdPT8HoIZ1QLmD6m2CrOQ7n7ZTOPV6jsE9Wzki9ncApRjHZlqexfkFOndmt4ARdiu2ZjjDb0nUfDsqoEtxI0wnvK3IZ4p9BceMsGzG41g1RkeJrePO/2L8q3Z64VUECEgtaON5PxShVrkpUvxQAElJsHdMdojDGo28o9huKX4Asqrr8tJi+HNhKQQNFQkwE515YE/KIxp2OVL74rByQTOslDonq0/Ndh7DvvqcRzKVyeuJ8o3o1prj0pfN4qd5LIyr1WYYkO1u+idfl1sB7d4NKOvTRvJdMG6osTu9Ysr/x7hPAOTeqU0ZGPTF/yUfNrzMJhE1drVbahK92II8jYhIu9tKl+ogw9d3uvl3KY6imXdMoqzg2xd7x/uv9XZ0NYnuc2l9ko1r+iydCim8d26YaM2dI2stdkJVysgm3s/GHQdQm6DfF2/+R0d7Q3FLtn4s3ZqTB6P+WYN0UI/vPf/85gQ0Vue4FU8/4R35Ix6TxLjb7/898BTKf9s5/q0DJd++KP87NBm+OoUt8gboWDgD8H7l4WmNKgl6F2rj0c/vXR8BgzCWPL9l2zFpMBVONC7BuR15OpC6oo1hPPTUFBrSYf4J2RUyMMqhbnoLxVbB1iv4JrvSUKDa0ovGZ4SAitSp6DBW20y0xPT+3lp10MKCOiFQg/WjMqBij2xpFCzrf99RUftmSNrIrIwYHi8hxVzMkJBEu2MPRsL1Tcratkj3bx3boGeTLyc0feb/shiRlsYHdMszX4pbZ2w6zaR5H0MujVKNfIiQGVFkHTfsoAdTKmuOTYkBpCr3mOXZKISgZQiv1BxUljBdNXL+QpPLUAajefVoB9vPh/H4YFIoulbZNNfrYvhNGUi+EMjF4rQECRnqJP7X4WPc6nMmH4XLzsPciuGyp+TaXisvpZLH7ASyBtYGU3ug+UZ35UtUFRS9etlWXoaJWKdiCoDnmd0Eu7vgZFSaL5RaJux9YtQKWdNobkw01ZypM0j2mHPo9J+4wGWtkOY8j8skGLohpZsNNQzaOLwPyL9UhrSa6HFZOEIlgV6Za9fGmoKK861fZ79tnPlFxgwFOjGBmxQlWKwo/LKdavVdlqP2e+y0YYalMNb4zlAA6VPJHOqe2TMeq7b1Rc6pdqr0rIWsAoLSB3ZpFNMhtgEfqZnslnrdISn0p3izSQrlxaDQoHGudTe5b5Y2NmtrnwLSosQRgsfjS1BhSGnmnxMj/g7MtEN8DYtRuTIW/wIFQRrlkrVCRNZowyFPJNDGWf7OWp+Ntd9Fi5zoZsVvTTqlf4qx1dqoqseNQS7lxuIx61wI1ylzCuX7c03phWfTge6UKFk6hcZ7pUrpNxkCmLqFwmL+n7Ng+s4xcKS68E+iZXMFRlVSmooUodVU2ti39u3RUK4A8wLiJD4sbkSv4tfF+qmqsaEpz2Zo1jcBnVDgfHEqzNrIN8u9paVtsD8Rz6n1k3hm/nWUhZyKXKuPwoLzYHMhpLqLRqSVmuWNlkWJIJg3UjY4v6JW+ZHaA8GUpT7Lpb22C8LTlyk/QSm5AIUiYu3wtYjOb6VRJIvc/js/I3uCxYa564xWOsMH+sEl+Fw0r9Zs0Jpj1vpKqNqXTi0LAlFS04s8iG6xyqSWASZkGxKwnuwiRsiThdJYcbqoSE6kHOKN5HK33Km2ZU2QeH+ix5sgGH5kS61eVd/WbFbKikV/Q4+ITvG3QGgBJsFVTfUfCpZA6Xj7woykWaxxSGs7j/CKOQquotCthTtJyKMGQnnRfdR2tJ9FC5yveAt1l4dwRVP09UcmcH7/GCqZH+ip91QT55tdWtRc2kMWJciT8b8OtKlcichZ6js4Jb4xv58bJgC73x6NRUSpILb9gI4p212sZmqsKB0BuyyOcyJoqpAChQCDzReRtF1ph5mSzGHSErkwBsLbeoCmk2RSnuZYtqO2iUTZDvsWaTUNeBRCqV7yqWHcBSigCgQhslRjtaE0qZxVwgYj42JDo4j3Pt4QWDjnmhJygry5WMQtCW0CLfuFrr6jyqhOaklJpCaJIHpEZQSItMcA61J+itcjAGBacivdRLycgg96JTK57AruRezPMOmM84IX1WqpbSOpqWsdab/nIEWhaYcX3KjWJrKoVs5us8QDfFQkDqaGpMQk3txFmglE6ZzdVzuA3bISGwm11tsdeTvNIs0L+mTeqFBKUrS2/LqD5XPfKCUVl8tUSrhWsgsaixIDZYh04vtm9uFPVyJHCS0iBZIo+foK3LlBZZ/Bi7OompRiYjBiY2UYVvUZhQ0YhKmESR68Odrq/MJYXxOmDVYcOtdcm3JsFP5SHFHPgFzhrQ7IkLWHWYMpNcKUSTKqW+FaRFgf9e1Z1ncx6rDa8j2iLCIBRN72QJMpznXFuW6Vn0eVoYQDWVVo36fvbSqaiZyH1xwH2TeTqpdtpt6rW73O0daboDPUVPSs+4qzMn5doLFaMsVn6tS3m4vAONyeVBAcq+PdC7SzucjyhZvpIK1isPeG9+XlxUOyjgQJMKNGW3gxbKSjP91AHtbIFyT35ZF3lMe9Q8typMYOQ73leYXrB+qLo81dN/MDbvi6ucdLAprrXSS/BLXy+0aHG1M3oq03ZFKZWlUuknwuBzpzomH7DlBjr4gjzpZ7I3EWMx47NF5KFZZdUXGbiVrPepnvVukPxccvo5i+Pv81xI8mLMQkTuchERPx8QnMWSo+ouK8vOVzHeE+18OqRBdIDiKg9QlkJfN2t1yNe1zByH9BXVO32uLLz4eudDwZB2NrvtdsE6tAtbnVRCGZZKALGq8VQCRFuQJM6Yt8i28mYX0sm484KaG1dtKAMAhjczl3NpTAweqiQEbYpcJkV9kbuLSpnnoLCS+CkbdI94tEuy7pRUo/1U3IneaCnnztaENBek51vxaYuIsFH1brlf6eAWu/sfJ2UOBwV91vQspArcGtNSQ6pIb+6W9MgtCXN/pLiE7lxVOlzUY6oIrhp8RfBWQUZOdFnjAxPIsznb9LRwEmmeZ1UM10M3RcybdlLGXkIHPchSh8OzUi8WxMirzZZTlwRcHpKdek6BJRUKoiiyZ8/cdBo6lQlVbDpV//arweuGOW5Ui+t5v3MeKadNJxyo5oiAwWcQhVfQlR6sAZPmksiIr+/OXX9Mk/XdtOIi35ItJc3JJ4TvwDya6Sq6P106v40zHXQmEoMF1M+smIs2sarTZx7sh2RcqXMVVIZG19/DqvZmIPHn73PdHVgBxGEcZDNjqFjwpxelCPxC0LFK1Tnedsvdq7RunCytG7e9Qu8AevvaOL/tsvbskyPyq4u2RTGRrqx5SkvFe2o2ms23sFJuew8xAuuZsR8mieFDQ/lUhHSlK6s3U50hNjgSGUZY3O04pMP53Moye7j4R3Hw7PB4d+9gKL6j88YMPgYKC/DwZFjUTvIZfhA23qlenlRBP7MxMcObKdeImeLPhWERLs16sfJszD5sGUUkMpXuC8eRAGKSjCXEt5jf+Wx8UUaT5FdVd16OTwErAMEDA4ZthmF5u0AFoJmtgKkGmf6w/QQ7+UaCFfsGit0BjZsBqrY+WVx4tISS9HxmayihAfmOwkrTK1f6K1oQrvbuedVWyQtRtuRGKrUpQQaiOKHr6uPW+djXfGePPb676Em6eKPYZaTOASwrQPv1swTD7BGdvax6c51bhlllQekAzp3nrEuHSF/WWdDqDqs7V9r38xiXXdoK9nn80Y2s/JocgAFvEaOr8TUda3BSvMM1Ah3ZkkwL6B0S2+I5RFw+zr0E2vqQxpmrvtj6F9nd2JvJVz5UPOll//mhoujc3BgN8xLpAa8FedjPUEmNL/erlc90B2U1ZpuBwgD1GADkh3A4TmVt0ywM0+nY70pdl78O+mW8jg3WqTB+XdWv5OcPqIQDudF5iaAojd+7K7b99fMJkjaNO7diHjR2/YzL8O+uoM4do2Oq/sspTzDSpKtF/1b4sy3ZgEptmhtokUBwVEjJCUaNTI3c1d9SvivXfkCiIE4rW0zo8B4VXwSofzagO0tRn4iqDQ2eLdqqaZp0iEHxTT/MwN0CcpW6A96+oTFgHcit0ZdwyeLbcUXxjMlpH0tbyKA5ol86kSBRlXaD6POfKOtBHLCpH4Ioj2xpPvtQbduh+I3MBdBJnD90O7XYi5wpf2wKo65jyxRJqZon3eaA/xIHk3dJVQ4/ow3y08kyy0E2xioKK8MZaiN2ESKWIsDpJaaKFN0nkCUJW7J3bTN4SENJdQaApEQqiAAkbdh4zvKymWODlRhuynGXSSGfd4i5yW6HZaknb+mkTOtSnviiTTnl8JidGqxWJVIrsBRpSz7VKVZNcKU/AMbUA4K1fFJhPKlByWbhYNnJUvhM32ZFTdcfDv4WZVHqWFpfPwexxZwGLcqn9qYiX5Bq7FYGcuHFaEeoSPuwMcoLj5MlkjRZp9IbmZ7XY7w4Lo61us3Ps7qlg6yWCoSjz+NCOdyONd1QfHlANXQ7NeFRAhRLAYrZRnmUNX3lJl6PLwjymuYYV0cAFT88Sraxs/5EE8TxqC7MZif7MkuFMc+k1VwcUynPNqdTx7ccl/InMtx75QawVWK3af5PFUlVBNfkIFbzGWSHfSj8l50aijEg1vC0nUUOHWZAUypEwWmShFL46e9mLhbPuUO9KzZfCqmKV0oVhQNzoYJ78BB5OF7wAHWKIrk+HG068Y1O4uMqOGFculNrzueyJfJ8u1kYOLz5hfJ/ylevzHIaNdXZa1mOEvVluqN2L897FLOvym/hmeURkjA27rRKi2nEBCMUFQ0anDquNmRrr7r5S95TxhMXUDY0slUjOISMwnjx2zmftaM2ZOSn61O89MPx7r4sNazLlQSy4llXuYQeP5fNdC4hIzZnElyv5BIykIlN/gj1lBvqVF65UWuW+am3RZ5sf3VcXDq0Pu8vo2xplVHSjtwPrx0sQd9OyZt7RRnhLXKN6HRM8OSMhZDqL0kpSrtftcG/tKYT/0QZDbZyY2liqyREYwaibt0wFZrM7ljf1XKtoqkH8iZ9Kx/0Luqdqpqrd+1Z4sruVBg2j3zrS3UpHU0FqdxVP09CyP9/ID9YoDjI2Mr4oKx59oBO44k+L+0bNfVNBa7OwezE5SxMX3RwqXHxkL48aFzAPftj2BeabXECZXjwTPL74vuiuhlkoT3j0H8UZqJIX1MCqS7jT855Lj39mqyjTJzy4WKxS3skrMW/yeMGpi5EZ2npXMo05h3nV8SSEoTH9INaMyD6cNQ7rQI7Kseeo/NEQpeGs0tu3Gm/pEIwkCmuGWr/4xriq1UjzSemjeLBs1zvk27kg9ktimrD1vmYTWiB4IQclyVQ4EXG0knMuIhkiTxfq7N2NB311RnildnVZQtM7i+qLuNVxVaI8BOSbYry7ExyEsB3i/+9h4IsEf1HJXmLRMt9rbLzjSZD70kGv/c0g395voUL0PDof8op8KTF6v2vOgV/OGCF1f6Iud7AdLkB/4ghTksI2+Je3Ran2F9hjnsr1iF3voQKCl6y5VUZmeLT7nwptjQErwF3ePgbqZO8JMAsXRkQXkI8mjYF2Dlp2JJjhOdoBJy4/EmcfMlJ3CdgO6RR5VH93EH5bZnHIspo502ODk/ys4OwavNh8ksmT6i1PY9m2gD5HT6uIZrJCUpbO+SjNPigsQZmov+4qGkodZqhFUzLUY8PvyXBdCvfd2oMAcI9F4bE2KaaijJhJssoKNnFndMrBbfgWrdp8JQ60M2aSRM7Ffv3UjrYf+2/AS1RfUQ=",
    "wm/planner.py": "eNq9XFtz20h2ftev6Mi1ZcACYVKyNR5qOROGQ1vKyJQjyZtssVgsCAQpSCBA4aKLpya1j+s8plI1lbd9STLevOZt35b/ZH5JvnO6cSV0sTW1rrKES6P79LnfWuvr62tHvz867r9tbIpf/vAf4t1+d7D3eq/XPe6/PxQTy4+E99SdWzPXt2I38MUkEfNgsvzZc+SlP3HEpTtZfgqEhheON46cC/PIufjnIPQmb+mJbq6t7WGEI7QkckJDbL4UgR2bYkfYT50oFlYSRa7AjH/9X0FfiE3x178I/N53eolviK54Z8Wn4ji4ssJJJLpJHPjBPEgi8dayT13fEXt+7HieO3N829FFGzMKz3USMXm6/OQn8+Wn0AnXYvrCcyJhB8kiMgR2E7uOH4sEMxwtP/7T+/6g18c3lk1bjYTcNwHuxDTas8TUcmNAFtmOPwkdgvnNYfe7vf7gGN8lPqZe/iUWi3D5aYKBCyukIQph5poQDZFO3j06Onj/bn+vfyQWmMwCHJg/dOxTJ8R/bGL5J2GfWheJg3kiACneLj+CPG+KIGpRMI3n1rVOINI+FkHkSkI5wvZcW/QOBsd7g/d9LC6Edm2IG100vimAxSgJneX/BID40rUEiPcudCauHQeh6fpnjh2PaRmhuROgyyWIAJv3FDchYccPwrmFiS4Sy58IoDkWRFZQc5KEIKXOG194SQSahJGIlp8wB0gVEZTgDOAJs19aoYvpImzdDjC1hGzu+qCD8ALbSq51Q1jTKdMkEv2B2D84ZiTPQmtCpDTEInEj8d37wx4h1ol516HTuAQLuFM5fQ1X7xB5zwJsDDRoYWWJXwIPD+aO63mA3OF9DA6O+5LYRJaURwD0hrBDV6JHA4z0se9eOuDFRRiApIYgLrdEvPxzDJa6dqMYzLohlp8Ane/MiRWBM8+5hBTZwXzhORgHSVDsxHvZwPrvf9fvvj9efhSaB/6OrTiHgvDuBXJxmyjiiLOnliuwgp3IndspWLR2//Xr/jHNozYMhvNn/J1H39qnQeToauEjWniwdyi00IkiZ37iMR8AERKIi8QVFgNrA14AQ/yIO5Ju2ukvf/x38R3xL2aYWXNHBJdOmG5OX3sHsGOGG1u3QsuNAE9b9PpvxXMIKvjG8pafrDhwsTzBkEoKMQDWBOQx6ASpMDIhA/85EfTPPrNZGCSYoUiyVFoi0TTNXqMlOuISK2P+OASu1DvJBcSkLkgLbTInGMCLmbD18KHkKEd0e8fv+/v7fWAs56S1hWf5kgEVj/Y2WlhyY5cX5S8jsfe2+2ZvsPwI1t0Ru8RfEfFropBkx2BCcQJRaEAYIsvDlzwK7FjkZucai0bm2jq0uztfBBCu2J076TUU4uJG4Ct/kb0OgElD/jJ935wmPoONJTDu9do0DOaC1btQXwy6a2v93433e4bArzcHAKX1lSFar8Q9/54U+T1KyU9CX8f7Bj0lZhkzs/z1L2traxNnCvLcjM+dG80yxPWN3hbi/mUxEVgfaI6dhge+hWC1FXUM0p6LAMglFU5a03sanJw5pA+hrIRmWxGs0vKnTYO4cy41E49g7rhs6UrRxEnoQ0/6sWaBP+j39c2wNdLF8+diM3vQVA904U6FHCw6HbEtSH9e3wiwhx8AxwEEyPGwdDqjrrZvkW4eL0I3CDV3DqDwYXQXGp6I2TmpfWiiE+vE9dwY2yIJhxa/Xjgg6JSYDdImMYQtExpSBAG05U+CsfBq+dMriRalGLCFmQP5jyUo6/hi3WDYddoIXbRTzPzwI3+yANkWMfO9ieHQvgndqbfTIBRMV6BG7qy9lu5jgWFTL7BibWENrREj0BK/BT18PNEltppmM/uA39+CW8i/eJbNFw+1Mr1e6eKZeAUxLlPt1UjPZgfcwzIrjgDgosgMGFImGqm5iFF13tm6m2a9/b3eETHuu4O9wfGR2N/7h0PoBnCutvkV+1AG6d6ErCgYEn7VmZNcg2pR0nr5PGy1vOexv7VNLE0GN1amjz028pMeSj8BilTHTG8l8nDEwGOZhXsNjcHqSDGbJXUY+McJ2ZUigEjJnTN3SX0whaomSy15JecTECZlleEoYxWb2ERqrjhYnIOQwKwOxwUK04nwzIOh1fSch6Y0IUgqJ50O7ZFphTM4UZqu8/ymtSCXTGN+sMVvck6Y0o0UY7vEIlN5p9dTfhEEnuZZ8UMUlSK99ssf/tvYfmFM2FvjO1xKT4t1UptMa0S+rXRGpA1mJeYHCdl3EvHnIqIb3w2LkL021UQfHALLlBKgm3PH8rXGJjTXxJ13Gi1SN7YHxSD2cvMiEQnbUmNRtYk7JeqRm2jBP9BJOx5VHL7dzDxDC7NVDcnGkc/nXMfsMoBrdgCsfxm4bHshsQ67BtIfzuYzycQROITl8RiuYjwea3AAwKS7nRcGmY651WmaXxviqsPsmvNBZGJUZO7SDx4HvqDv1FeFcVd4Q+ZXIwev0zKbhvBs+dsPLuWFM51iHej4uTPHxUvMEsg3CBSseWeLL8GKdMV66yoTL9ZaV0x3Hqz07hSSkXjSMZQBUuqyRyQ74fLPsIm5anSvAaYUBLhPcOO07Rdgz+0X4OrIXNzQPhc0RqOh4Fe8SUlvyIe/KT5byzCrSEOInYYwxtLDwk8yEucVw/NEjREaXBxiYUyZelXKz9nhzzHA0NM56GYzV6xzgAn67GD5S1z6tPrcJLdwDscgjCCqJl5BwnOxDvPtR+PY8SMYR3+BGysMrRtNQkW8zd918Es34S7OtHzZKxdxpvKCgjFFFUXFIZnhg02I3LeJW0znxsHEmHRuelAKuHyofCvkTPTS9ORITkMzIskElzREq7I64jG58gIhmorOtDt2TYiu3bN56TpXWssQPd1YewDId6yhuODuVRD268z2PfGNaEqWJ+av7C/yJ4SDlTcFK0UG6NSxQhig1xamebBWVagvyBHi9v7gu4J33xYfsHLEsQyYXpwEyQx6bQOS689IA5UCuxW8TWaAPYUT8rFOn6WGcke+xg9shn4WvDxyTsiHmczYE+xR8CBxBLcDQqzxE31lQc/OmF7RZTIr0gHX8c3C6cghLNhbm/qQ1hytTAbcM2vBZMD6TUM5jHRdGhp8cMIgGnvuOeyGrat9EYQPYG1FRSZ/O+dkdbFBq/OChmj3amBLOYOG9Ubii/49kTHxUxdOCTQatvu2u0cs0B0gCLbOAraZCNhbFJgWonu9wqXnl7kM2vH12LYQiGqkG/ItGOnWyjxfHEQeJQFVtnxtuEKezYb1/aAPHweMQjEupa0qcJBjJYWFVPzk6hbJOYWksuRIiC3E7uDNODhf15nzAF6HiHIf9g6X//a7PuCDvEQJYnrGJaPIocjpTfcNY/I53bzrHyLYP14h5AeYwf0mEx6I6LWZsUcpq5FirTzcEX7O5NPE8zTSJy3SJ4ZotG7ndtJAqyxoTYjJm4QWxVMlKyyfKQLVi0lFFbnESfHnKqMnaWQF7/5b8ug5zCC/ST6LEtte/hyJE0prLf8vZqe/hQCEXvo0BvdNvVZzX9xiIHLkkeh278EcfBubUkRAWO0qN+SGg5Ib6VpwxilBK4VhopGUGEzui3xMNA7j2TAcFRCs0EyGiP35kPx5grKpr+KeaNQWj/wH3Ps28Co9KDit5HPBrkUxvCRKHyOuJ/f025k1A188F98iQn4sEu4lSHz+cIJkyIrPGVtEzEGXTUSd6pxckWRNKF9EYewQ4DRHuetHEW/5GWuTmUXRtqaxYmnwBAh3nkHuZKBAsLScxqtbtY0MFMfWut7+EiJ1e1yE0Lgk0NJlhkglNacyfMyyF5AG13ZXtn66KOtpxDIxgbjyqJpwWTF3JW1VY4cR/XsIPL0pL8ihvvqmyOvg8tOFVGt6DZ2kOlfMoTL7CNCq9Gowu5QHxTWDbllimmnT7OtpzdcP8mIRYXDcbiBwkOF+RITKYn+ZP5N5Zxm70fBCaEUmvcfsxo4+fnKwcs3Byg2bicU1gniNnEr5ML3L4xOsNb5y3NlpHJHXdU2hjjtD9PWyhvMA9ZGxi2CjkP0lC+z6CRw/ypEF7iSCAg45y/ei+fW22lkG9mSTyMTRVAOrDU3ThDWntBGJB1hEY+Dzd622epdNoSLxMh0amPi50Dbh8AF+KWtMi3yrkR2EDm3S4pqM3OyJ5Z93JJeR3VCBLis6VbmhHQ+6OmfV0u1jqxxX414Xw0rkDk6dUErBLWy7EB1ZSRyAkrG2bicTa72srU6YjVrbCIp9mmvSYfKaNIS8W/mNkMUi+oDuTdjUsdR/UJ4LmaS80di7hHfyqhKHKfxF5lhipIoOiYkC4sa3Yk4Nzeefc7iyK0NQ4rke5wl2xBEeya/TCC2Pvu3snST5NhwXKPvxomqMeUwGLxilwLzXN1RVsPV6li371CUHJk9ppmsqzEKch/xo2GYWJUebvzJE9hjcOSIblC9X9X8h0rnPzCrYhWg7sl4TObJSkaciPtzi35nQ1qA6NsOOm/wPY6P8wf2Hjc+WOYEqgXGFAexgCjKz9+mrUsGmUq7ZSWtAGrvhkH1dREEhacomnZ3/POcAE3xGFljmW3b1R/smJXCzElBa50npEZVk4QPRm5Bxnxvi+gWqt88Y0+xK3IevSPSoWHVWNfVnJM67lKZoixN4UeflAZTF+VBc7FfDjEokZRX6rJ52tlHOmDBaStJAnOLOmeF3FPMU3qauWnCmuXOdR+USh9XyHe2OhPhVqZ3LONyJmKtjq/Q+dSe5QsHNrwrKE5VFpkgkpKxEcs1slxUvczjYWivr5c7mMJmILp1LrYhK11fQGQT2iIVXuo6bepZl1B9eIpRWrF9QumEBDF6zpEQK6iPXHgYjlQFrtEZlAjtT8o00pnODZ8/A5Iy5ptLiKlFODCQ/oZ8w3PgFQxdD5rSyj7zNcb5FZc1EJru5zk77yxannXChgJbHzCDBPEoTz57dcS6lYeGq64jTy4VHbw5GMgdNMNgeltLgUXS29JLBKLrmlMsmz5zXGcrbUQ1R6YUG+q1QTrk2o9IKpSSdFY0p453FxmzyI/OKbL22rl41cxj4yW0wjCd3QMGZFFJYZKPEpROyxj5JYGHnyz9TwTZw/SKcZPhrM3D0IofIDy4ZoBb4IQ3FZlEyB2dprSb8tEvx9zyXeSyd5ucCj5+oHoyEfKiz5Z8a3H5RstzwOGqXl65IuvzcmRfwUbM0DS8tTTgIKXFARaFC+wVbEijK3PWOK6lKmmpMOaG0rvdZiiNrDnnXPRT/2D8+GBxlRUoZEUSZVqNapUyzzoNJ2pWTNkElfjW5SuiK65EVV1gK+EpZ6vZCOIJ6i1qFZP8IlJoYPN17++7g8LgvDpb/WewPcgo9YJrrU6uSAlP2hIBhUxNZSRGy7SvW2WiiPL5SioQoGJvHDzIL35dXyDlEigxWNCmGQHR+aXkIZTIldX9SVEZmnPVI0UMOdhgkl7JNwJNdaAi0vy37n5OrFcLk8k1plbupcTtMqQfa5r6xlKcTlbJJM40EFh7W5Rpv0r4aUvXFzIbNSQ3tplWX09Bask7REPmIq7oRJG4yR1JHFbn1Udk22UovN7YMsaUTWColUEg2kK1ifyrLBXHoVHFEZuwuy5olQCvV/HbLX9YXANbyqg6pAWgVDZxCJD3nAG0WrMs8SIsK3ClFz4mWuI/Z78VdzKV32rIJFTOPStZccq5ejXg1WvKZmOkmrcqZDJogqzS/Ca3JO/gbvhPmlWZuuJQl4LTVjwN00iTF4nLWF4mZgmThUc1RyxsPVashlaq4B2uShLZLrWw034ZQTYI2l7nvLS1HjjOJOtQNQAXJTmvTEF7YaZov6Pf4+kZWfIGWhSwDr5Si8Xschy7C4016fzXmrJH8LDoNgrjTLFWrWaUUCvFaoViNzzlpx8XsIy49EFR04YXy55jTIiYBRPGE2oACn2BWcEuYaTIFEUarqwIwDCCxDv9+pNd5DLvRfS92u0fdw+9KZmNDDGSpIaIWQGkorHDusj3hTqWIW4TciUX1aCgiHuP4sNgcOxWr+BLbvB159QiQF/BPLdlG1WZlJdumpL0XThRZN1niy4mVX82BZQEi5b+y6ny0605OetZ9SQytpZ21lDY9tZIJ99PJ/jXqJn7XK6bQ8PVt9X3cXVquZ1DHW1SfbTJE6M86/sKEEpoEYEzJ2GTqTgI/idSgqRXFHVUnWSuFicTeO9Tnk/Ya1DYa7BDfHhmqKyBnd3yHR3kCx4rOKyWjQdfgVJrzolSm3uGxw6Elu85In/FeSRU2xW+5t2zQHZEmz3vKgAbICWX+VT81fauNbUOMEWBrkKBrKnX7jK8RNU8NtS2oByjeUdF8bmfLtdM5NzDpgudb8MtyyxjXzxdsbuXs+KRce91gwZDpSraOWadV2nuVAcCddbwJK/X47sIB23ZL/F1HbFMSSRtq29zwxh/JXr0UoNLWVBW7ECX4DtUph3bevSVBcfO2zme2nu6SGYmRKEetuHNSCbzrHon+oHdw2Bf9o6Pu75cfj+6WvhOHPeBaMDTmWVrzhx91Nn0FwNgIfiMqRW8bguYIMiTUCd/O/BcZioC9QkoxL38O2YjBu/TB6H5SCGVdSiDW9nV+QX1Lttm1i2pyal0GoSu7mdOySeObzCaCppQNLvIn4CgnXSVaVEhaxBGi1YCsShQACxONwTYEENaBx3MysYTdFg1MNyzgcQQXaKtcrSpToi3XK31DiS56WEMU6qraKFitZ6K6YHkvkv70a9jeJJYusgJtBxiwUy5Mh5Xkl7QZiwUh6pEFSUmwXbDt3jF3I6SHDpRh+a3YaorfpAcSyLJxAv/oJoqXP0Ohb65VK8DUoMPtOcMqHip4HoF8vJWOuOAQQqeEezE0BKtSu2fT3CL3j4ZcsPcGtLecxtd6FSntXycZRSeTln9824ct11w+QETnhziaTBtJNwgdUvYwwOYDKmT93w/Y/COCKzc3RXYJM/cJuuSppvmSWnBuU076Cj5l0mcLH8EqsknUKFyV0qxXk6iP5Z5yubR3MHi9xxz0y8f/Ei9XmIYz+VgSPMMlrhR9GtW/hXK9RHutrqngs1jKVW22F3l3LcP58qUESamlILRsaMyzhO7gXVtulANF51vs0KJ2x5ouNeDuYuiO6rh1fOoyw76sZ1iW2ruIT5vA1CkPfAPPwXzRZvgjK5kqoWQVjk0msetRn8Cc1J9XW4mWzEfQQkvaSszAJoVWde5OaZJRpXKIAgEIxPAMg6OdB7qNBGd2hE0eC/HdPHjS/rXVXP4kY+vQWlA7eF15TSN4YgRSDvdCX0pWvyQ6A0rZfd8sttc3V0vp7FoY0mZwDa4jQ/EUC7pyC4GKHAauCEn/jcTH57i42rRCtAWJd8S//D4fnofRBp24KIwuVW3cvGpz9JiqzZPcgMrza1EpHqBOHRUMqJjTSjgLW+knIZBOS4Wk1cYlLt52wLZki1x2s7RTQn6Tke+qYxD0WlcnIeBqDVMVRASk93gGPmLaqa/zIyY01j4NgDHtPpdYT9v/VgDdbw5dQ5zCdR+RP7u1MgDk4hGjau8i9oclpRu5Ard8rFMq5M6UBuelWERKddHHUVcFULKzgQKtyVMghbKIZHboYFp2JHCtgoumrDQYvG11I/saACN147Ta3PjGty2+zabwghkVPqi8adoe9qHpZuhcJHQEjtuTx9pxmFBraVLMEruxppFMPBfbW7rKOkGPbZI2+/qVXj9HHt7wkQs5Ga4RW3Un1lwb/rDO/dfRelsMJWCAe90L19ucX/jREMURSeklxTYEDST2xY+jsiiOc75XaYsvE0ju7M00nDVffvLgR8uzWnmSqKxuZXtEtR9GU3jf4OhQpvooKaLK1SyGvJdynSKp5IApIJSHV+7sUdgBmSMicgMqka1UaRqQgJWaakeXo80Tyz6nCi89oBFQNov7Gtm/WMXxedpDqb/yw7SFgr/E5SngIQVVxl5qvD4HdQdVzSCF//7O5jqW4JNs0dNur3d40NvtkwaWzUkTdciPZZqTzHRMtwQIw3sw1LS0k0d17o5EQxykeWGiGVUFaaMup9xHdXn89Gxh8RAWNLQ8Uw5H0l/xDFWarf1Iz1C6dliMhEy278L18R3VxEtBGx9VlZm1zWYzy7VlxmrV+bLdmuMWVZ0tPV7qxvrgdDS1HZhmvXxYgKFMPb/aZO/yT+JVbml9dmi5PsGFrVUbZFtVDhra1ElUcVMruSCbyV39rMXZhzSDc9cU91imFUkpFAfkM4r4h7Y7GqWiUhhB7A8A5evSnLItib5faUtK9VBaTM500WsTjDw+DagTSa486BZOA1XbqCqMKT3nR9rXJ6LXpZrb2/7guLtPfeUBSUGcn/jSMkeqppWRDw/QwYEyvVbTA3KD5HE0R7QzeZW69ivOYJkpssNQK13It0LUKOSan1WguwemLKb8IrBOVMRVDBgqBwZU6KBU9PBklPUrGEKr08sYwket5CD2Gk/gAEbmkXIbq4yED0q8hCm3eIeFtXKoKHqQJ6DVx4wGML/BrdXq5C09VTQrPSn2HhQjFpWWLUQn90QitEsqiFx0sk1kB0cNaPkfztvqmHBMr2Rrh357GezH/ORkr/+2Ws5a1P+lg8pfNPjY20UYv7e/z6X00l80kPlw2aMqT6HrYiet00b0FzhCzvNh7XvLWH5nGz9DOvITdaDiHA97oLLWZ9SrvqxM5dMPuTBd8cJGLjtGqQKFL/wUTAVkBk8Gy9+2nPGIEkahflEpXTwiK39baUFOSZUFStlL2bi7ZjFsb74YqcIFfaJMXyGD+j39wYvMwO/wuXygS9ZasKvvKdNiUgTyPd6qXGvpaNJdCe+1Xzs7VsmKF/9aCASTgrmTIJzccrjgoSnwz8yQqSQhcEeOSUdlGfkmTTleAH8XNSHB3ypPfv9x2JX4TUpoTfrCnVxL/ESwAefacFjINXxviEVnMTwd6auJkNHqGr4+KvQqa36xyyAzLCuO3FBm1M5HqR/IR2fC4Ere4YLuAeaoPmtUCglum7rGV3zQMg/0HO9z5T7Di7vdk3u4Z6XypNgTcFryqG7DZq3fkh/s+hwXqrJ42XX6gvUJpZPrIZwn/jMSFMkVjJKv539Swl4k1BBKf0ZHqzlvtWBhJl9qwZL8Fa5yvsfVCRX/EjgyjkddQKcAEatBnc7i0873lG4g1ep4ddKwCvm9Xl9KaUd20aV/0KPo+4hvZIWrOWqnylor+0aEGvLsvsATKua27Ty1zQvSwbNCUa3oxNnNsq+mAJRuWr0wqylH5LLx9Sbw9f8a3Za1",
}
D = "/tmp/arc3code"
for p, b in FILES.items():
    os.makedirs(os.path.dirname(f"{D}/{p}"), exist_ok=True)
    open(f"{D}/{p}", "wb").write(zlib.decompress(base64.b64decode(b)))
sys.path[:0] = [D, f"{D}/wm"]
T0 = time.time()
class FairSem:                                # verrou ÉQUITABLE (file d'attente) : threading.Semaphore laissait 7 jeux sur 25 monopoliser
    def __init__(s, n):                      # la carte (un fil qui relâche reprend aussitôt) -> 18 jeux affamés (30 actions en 30 min, 27 oct.)
        s.n, s.c, s.q = n, threading.Condition(), []

    def __enter__(s):
        me = object()
        with s.c:
            s.q.append(me)
            while s.q[0] is not me or s.n == 0: s.c.wait()
            s.q.pop(0); s.n -= 1; s.c.notify_all()

    def __exit__(s, *a):
        with s.c: s.n += 1; s.c.notify_all()


GPU = FairSem(3)              # au plus 3 jeux CALCULENT en même temps sur la carte, chacun son tour (file d'attente)
NEED_GB = 2.0                             # mémoire GPU libre exigée avant de charger le modèle d'un nouveau jeu
STOP_SEC = 8.0 * 3600                     # marge sous la limite de 9 h de la compétition
LIGHT_SEC = 5.0 * 3600                    # au-delà : réflexion allégée (4 itérations au lieu de 12)
LVL_BUDGET = 100                          # 27 oct. (user) : au-delà de 100 actions sans nouveau niveau, le jeu s'arrête (score ≈ 0 de toute façon, temps rendu aux autres jeux)
W = dict(crit=3.0, lc=1.0, nov=1.0, eff=0.2, mem=0.5, go=1.0, dream=3.0, dist=3.0)   # critique ×3 + distance au but (étude vc33 27 oct.)
CKPT = (sorted(glob.glob("/kaggle/input/**/wm_seq*.pt", recursive=True)) or [None])[0]
print("modèle du monde :", CKPT, flush=True)


class MyAgent(Agent):
    MAX_ACTIONS = 3000                    # choix user 26 oct. : les actions en trop sur un niveau raté ne coûtent rien au score

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = abs(hash(self.game_id)) % 100000
        from plan_agent import PlanAgent
        assert CKPT, "modèle du monde introuvable dans /kaggle/input"
        import torch
        while torch.cuda.is_available() and time.time() - T0 < STOP_SEC:          # tous les jeux cachés démarrent ensemble : attendre la place
            free = torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved() - torch.cuda.memory_allocated()   # la réserve inutilisée de PyTorch compte comme libre
            if free > NEED_GB * 2 ** 30: break
            time.sleep(20)
        with GPU:
            self.ex = PlanAgent(CKPT, seed, "grad", H=12, seeds=8, iters=12, w=W, focus=True)   # H=12 + critique ×3 + FOCUS (meilleur réglage du 27 oct.)
        self.n_act, self.lv, self.lv_at = 0, 0, 0

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        if latest_frame.levels_completed > self.lv: self.lv, self.lv_at = latest_frame.levels_completed, self.n_act
        return latest_frame.state is GameState.WIN or time.time() - T0 > STOP_SEC or self.n_act - self.lv_at >= LVL_BUDGET

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if time.time() - T0 > LIGHT_SEC: self.ex.pl.iters = 4
        st = latest_frame.state.name
        grid = latest_frame.frame[-1] if latest_frame.frame else []
        if st in ("NOT_PLAYED", "GAME_OVER") or not latest_frame.frame:
            with GPU: self.ex.act(grid, "GAME_OVER" if st != "NOT_PLAYED" else st, latest_frame.levels_completed, list(latest_frame.available_actions))
            return GameAction.RESET
        import numpy as np
        fr = latest_frame.frame; anim = len(fr) > 1 and any(not np.array_equal(f, fr[-1]) for f in fr[:-1])   # effet visible seulement PENDANT l'animation
        with GPU: a, data = self.ex.act(grid, st, latest_frame.levels_completed, list(latest_frame.available_actions), anim=anim)
        self.n_act += int(a) != 0
        action = GameAction.from_id(int(a))
        if action.is_complex():
            action.set_data(data or {"x": 32, "y": 32})
        return action


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway sidecar to be ready.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the framework into a writable location.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Drop our agent in as a framework template.
    !cp /tmp/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register MyAgent in the framework's agent registry. We rewrite
    # __init__.py because the upstream version eagerly imports
    # templates with deps we don't ship (langgraph, smolagents, etc.).
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")

    # Point the framework at the gateway sidecar.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run it. The gateway records every action and emits submission.parquet.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Save-and-run-all (commit) mode: emit a dummy submission so the
    # commit succeeds. The real submission.parquet is produced by the
    # gateway during competition rerun.
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
